<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/PointWorld_compute_depth_and_extrinsics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compute Depth & Extrinsics for PointWorld (Standalone)

This notebook is **self-contained**: it first writes all dependency modules to local files (using `%%writefile`), then runs the full pipeline for:

1. **`compute_depth.py`** — Stereo depth estimation via FoundationStereo, saved to H5 files.
2. **`compute_extrinsics.py`** — Camera extrinsics optimization via VGGT + robot mesh rendering.

> **How to use**: Run all cells top-to-bottom. Edit the **Configuration** cells to set your paths before running the pipeline cells.

---
## Step 0 — Environment Setup

Set up `sys.path` so the written modules are importable, and create the required directory structure.

In [ ]:
import sys, os

# Directory where we write inline modules (acts as the PointWorld repo root)
REPO_ROOT = os.path.abspath(".")  # <-- adjust if needed
REAL_DIR = os.path.join(REPO_ROOT, "real")

os.makedirs(REAL_DIR, exist_ok=True)

# Ensure repo root and real/ are importable
for p in [REPO_ROOT, REAL_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("REPO_ROOT:", REPO_ROOT)
print("sys.path entries:", sys.path[:4])

---
## Step 1 — Write Dependency Modules to Disk

Each `%%writefile` cell writes one source file verbatim. Run them all before importing anything.

In [ ]:
%%writefile real/__init__.py
# real package

In [ ]:
%%writefile real/gcs_utils.py
#!/usr/bin/env python3

# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""Google Cloud Storage helpers for DROID data paths."""

import os
import re
import tempfile
import subprocess

def is_gcs_path(path):
    """Check if the path is a Google Cloud Storage path."""
    return path.startswith('gs://')


def parse_gcs_path(gcs_path):
    """Parse a Google Cloud Storage path into bucket and blob names."""
    match = re.match(r'gs://([^/]+)/(.*)', gcs_path)
    if not match:
        raise ValueError(f"Invalid GCS path: {gcs_path}")
    return match.group(1), match.group(2)


def download_from_gcs(gcs_path, local_path):
    """Download a file from Google Cloud Storage to a local path using gsutil."""
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    
    # Use gsutil to download the file
    cmd = ["gsutil", "cp", gcs_path, local_path]
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"GCS download failed for {gcs_path}: {e.stderr.strip()}"
        ) from e
    
    return local_path


def get_local_path(path, temp_dir=None):
    """
    If the path is a GCS path, download the file to a temporary directory and return the local path.
    Otherwise, return the original path.
    
    If POINTWORLD_CACHE_DIR environment variable is set, use it as a persistent
    cache directory (under <POINTWORLD_CACHE_DIR>/droid) instead of temporary
    directories for GCS files.
    """
    if not is_gcs_path(path):
        return path
    
    # Check if caching is enabled
    cache_root = os.environ.get('POINTWORLD_CACHE_DIR')
    
    if cache_root:
        # Use cache directory - ignore temp_dir when caching is enabled
        # Recreate the GCS path structure in the cache directory
        bucket, blob = parse_gcs_path(path)
        local_path = os.path.join(cache_root, "droid", bucket, blob)
        
        # Create directory structure if it doesn't exist
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        
        # Check if file already exists in cache
        if os.path.exists(local_path):
            return local_path
        
        # File not in cache, download it
        download_from_gcs(path, local_path)
        return local_path
    else:
        # Original behavior - use temporary directory
        # Create a temporary directory if not provided
        if temp_dir is None:
            temp_dir = tempfile.mkdtemp()
        
        # Create local path
        local_path = os.path.join(temp_dir, os.path.basename(path))
        
        # Download the file
        download_from_gcs(path, local_path)
        
        return local_path


def enforce_gcs_cache_policy(
    scene_paths,
    stage_name,
    require_cache=False,
    allow_streaming=False,
):
    """
    Validate cache policy for GCS-backed scene paths.

    Args:
        scene_paths: Iterable of scene paths (local or gs://).
        stage_name: Name of the calling stage (for logs/errors).
        require_cache: If True, fail when GCS input is used without POINTWORLD_CACHE_DIR.
        allow_streaming: If True, bypass require_cache and continue with warning.

    Returns:
        bool: True if at least one GCS path is detected, else False.
    """
    if isinstance(scene_paths, str):
        scene_paths = [scene_paths]

    gcs_paths = [p for p in scene_paths if is_gcs_path(p)]
    if not gcs_paths:
        return False

    cache_root = os.environ.get("POINTWORLD_CACHE_DIR", "").strip()
    if cache_root:
        print(f"[{stage_name}] cache enabled: POINTWORLD_CACHE_DIR={cache_root}")
        return True

    message = (
        f"[{stage_name}] Detected {len(gcs_paths)} GCS scene path(s) but POINTWORLD_CACHE_DIR is not set. "
        "Without persistent caching, repeated stages may re-download the same objects and increase GCS traffic."
    )
    setup_hint = (
        "Set a shared cache root before running this stage, for example: "
        "export POINTWORLD_CACHE_DIR=/path/to/fast_disk/pointworld_cache"
    )

    if require_cache and not allow_streaming:
        raise RuntimeError(
            f"{message} {setup_hint} "
            "If you intentionally want to stream again, rerun with --allow_gcs_streaming."
        )

    if allow_streaming:
        print(
            "WARNING: "
            f"{message} Continuing because --allow_gcs_streaming was provided."
        )
    else:
        print(f"WARNING: {message} {setup_hint}")

    return True

def list_gcs_files(gcs_path, pattern=None):
    """
    List files in a Google Cloud Storage directory that match a pattern using gsutil.
    
    Args:
        gcs_path (str): GCS path to the directory
        pattern (str, optional): Glob pattern to filter files
        
    Returns:
        list: List of matching GCS paths
    """
    # Make sure path ends with a slash if it's a directory and doesn't already have one
    if not gcs_path.endswith('/'):
        gcs_path += '/'
    
    # Construct the gsutil command
    if pattern:
        # Add the pattern to the path
        search_path = os.path.join(gcs_path, pattern)
    else:
        search_path = gcs_path + '*'
    
    cmd = ["gsutil", "ls", search_path]
    
    try:
        result = subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        # Split the output into lines and remove empty lines
        paths = [line.strip() for line in result.stdout.split('\n') if line.strip()]
        return paths
    except subprocess.CalledProcessError as e:
        if "No URLs matched" in e.stderr:
            # No files matched the pattern
            return []
        else:
            raise RuntimeError(f"Error listing files in {gcs_path}: {e.stderr}")

In [ ]:
%%writefile real/real_utils.py
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import numpy as np
from datetime import datetime
from numba import njit

@njit(cache=True, fastmath=True, nogil=True)
def project_points_to_image(points_3d, transform_matrix, extrinsic, intrinsic, image_width, image_height):
    """
    Fused numba kernel to project 3D points to 2D image coordinates.
    
    Args:
        points_3d: (N, 3) array of 3D points in local mesh coordinates
        transform_matrix: (4, 4) transformation matrix from mesh local to world coordinates
        extrinsic: (4, 4) world to camera transformation matrix
        intrinsic: (3, 3) camera intrinsic matrix
        image_width: image width in pixels
        image_height: image height in pixels
        
    Returns:
        projected_points: (M, 2) array of valid 2D points in image coordinates (subset of input points)
    """
    n_points = points_3d.shape[0]
    
    # Pre-allocate for projected points (worst case: all points are valid)
    projected_points = np.empty((n_points, 2), dtype=np.float32)
    valid_count = 0
    
    # Process each point
    for i in range(n_points):
        # Convert to homogeneous coordinates
        local_point = np.array([points_3d[i, 0], points_3d[i, 1], points_3d[i, 2], 1.0], dtype=np.float32)
        
        # Transform from local mesh coordinates to world coordinates
        world_point = np.zeros(4, dtype=np.float32)
        for j in range(4):
            world_point[j] = (transform_matrix[j, 0] * local_point[0] + 
                             transform_matrix[j, 1] * local_point[1] + 
                             transform_matrix[j, 2] * local_point[2] + 
                             transform_matrix[j, 3] * local_point[3])
        
        # Transform from world to camera coordinates
        cam_point = np.zeros(4, dtype=np.float32)
        for j in range(4):
            cam_point[j] = (extrinsic[j, 0] * world_point[0] + 
                           extrinsic[j, 1] * world_point[1] + 
                           extrinsic[j, 2] * world_point[2] + 
                           extrinsic[j, 3] * world_point[3])
        
        # Check if point is behind camera
        if cam_point[2] <= 0:
            continue
            
        # Project to image coordinates
        img_x = (intrinsic[0, 0] * cam_point[0] + intrinsic[0, 2] * cam_point[2]) / cam_point[2]
        img_y = (intrinsic[1, 1] * cam_point[1] + intrinsic[1, 2] * cam_point[2]) / cam_point[2]
        
        # Check if point is within image bounds
        if img_x >= 0 and img_x < image_width and img_y >= 0 and img_y < image_height:
            projected_points[valid_count, 0] = img_x
            projected_points[valid_count, 1] = img_y
            valid_count += 1
    
    # Return only the valid points
    return projected_points[:valid_count].copy()

@njit(cache=True, fastmath=True, nogil=True)
def generate_and_project_workspace_boundary(workspace_bounds_min, workspace_bounds_max, face_density, 
                                          extrinsic, intrinsic, image_width, image_height):
    """
    Fully fused numba kernel that generates workspace boundary points and projects them to image coordinates.
    
    Args:
        workspace_bounds_min: (3,) array of minimum workspace bounds [x_min, y_min, z_min]
        workspace_bounds_max: (3,) array of maximum workspace bounds [x_max, y_max, z_max]
        face_density: number of points per face edge
        extrinsic: (4, 4) world to camera transformation matrix
        intrinsic: (3, 3) camera intrinsic matrix
        image_width: image width in pixels
        image_height: image height in pixels
        
    Returns:
        projected_points: (M, 2) array of valid 2D points in image coordinates
    """
    # Estimate maximum number of points (6 faces * face_density^2, but we use conservative estimate)
    max_points = 6 * face_density * face_density
    projected_points = np.empty((max_points, 2), dtype=np.float32)
    valid_count = 0
    
    # Extract bounds
    x_min, y_min, z_min = workspace_bounds_min[0], workspace_bounds_min[1], workspace_bounds_min[2]
    x_max, y_max, z_max = workspace_bounds_max[0], workspace_bounds_max[1], workspace_bounds_max[2]
    
    # Generate step sizes
    y_step = (y_max - y_min) / (face_density - 1) if face_density > 1 else 0.0
    z_step = (z_max - z_min) / (face_density - 1) if face_density > 1 else 0.0
    x_step = (x_max - x_min) / (face_density - 1) if face_density > 1 else 0.0
    
    # X-constant faces (YZ planes)
    for x_face in (x_min, x_max):
        for y_idx in range(face_density):
            y = y_min + y_idx * y_step
            for z_idx in range(face_density):
                z = z_min + z_idx * z_step
                
                # Transform to homogeneous coordinates
                world_point = np.array([x_face, y, z, 1.0], dtype=np.float32)
                
                # Transform from world to camera coordinates
                cam_point = np.zeros(4, dtype=np.float32)
                for j in range(4):
                    cam_point[j] = (extrinsic[j, 0] * world_point[0] + 
                                   extrinsic[j, 1] * world_point[1] + 
                                   extrinsic[j, 2] * world_point[2] + 
                                   extrinsic[j, 3] * world_point[3])
                
                # Check if point is behind camera
                if cam_point[2] <= 0:
                    continue
                    
                # Project to image coordinates
                img_x = (intrinsic[0, 0] * cam_point[0] + intrinsic[0, 2] * cam_point[2]) / cam_point[2]
                img_y = (intrinsic[1, 1] * cam_point[1] + intrinsic[1, 2] * cam_point[2]) / cam_point[2]
                
                # Check if point is within image bounds
                if img_x >= 0 and img_x < image_width and img_y >= 0 and img_y < image_height:
                    projected_points[valid_count, 0] = img_x
                    projected_points[valid_count, 1] = img_y
                    valid_count += 1
    
    # Y-constant faces (XZ planes)
    for y_face in (y_min, y_max):
        for x_idx in range(face_density):
            x = x_min + x_idx * x_step
            for z_idx in range(face_density):
                z = z_min + z_idx * z_step
                
                # Transform to homogeneous coordinates
                world_point = np.array([x, y_face, z, 1.0], dtype=np.float32)
                
                # Transform from world to camera coordinates
                cam_point = np.zeros(4, dtype=np.float32)
                for j in range(4):
                    cam_point[j] = (extrinsic[j, 0] * world_point[0] + 
                                   extrinsic[j, 1] * world_point[1] + 
                                   extrinsic[j, 2] * world_point[2] + 
                                   extrinsic[j, 3] * world_point[3])
                
                # Check if point is behind camera
                if cam_point[2] <= 0:
                    continue
                    
                # Project to image coordinates
                img_x = (intrinsic[0, 0] * cam_point[0] + intrinsic[0, 2] * cam_point[2]) / cam_point[2]
                img_y = (intrinsic[1, 1] * cam_point[1] + intrinsic[1, 2] * cam_point[2]) / cam_point[2]
                
                # Check if point is within image bounds
                if img_x >= 0 and img_x < image_width and img_y >= 0 and img_y < image_height:
                    projected_points[valid_count, 0] = img_x
                    projected_points[valid_count, 1] = img_y
                    valid_count += 1
    
    # Z-constant faces (XY planes)
    for z_face in (z_min, z_max):
        for x_idx in range(face_density):
            x = x_min + x_idx * x_step
            for y_idx in range(face_density):
                y = y_min + y_idx * y_step
                
                # Transform to homogeneous coordinates
                world_point = np.array([x, y, z_face, 1.0], dtype=np.float32)
                
                # Transform from world to camera coordinates
                cam_point = np.zeros(4, dtype=np.float32)
                for j in range(4):
                    cam_point[j] = (extrinsic[j, 0] * world_point[0] + 
                                   extrinsic[j, 1] * world_point[1] + 
                                   extrinsic[j, 2] * world_point[2] + 
                                   extrinsic[j, 3] * world_point[3])
                
                # Check if point is behind camera
                if cam_point[2] <= 0:
                    continue
                    
                # Project to image coordinates
                img_x = (intrinsic[0, 0] * cam_point[0] + intrinsic[0, 2] * cam_point[2]) / cam_point[2]
                img_y = (intrinsic[1, 1] * cam_point[1] + intrinsic[1, 2] * cam_point[2]) / cam_point[2]
                
                # Check if point is within image bounds
                if img_x >= 0 and img_x < image_width and img_y >= 0 and img_y < image_height:
                    projected_points[valid_count, 0] = img_x
                    projected_points[valid_count, 1] = img_y
                    valid_count += 1
    
    # Return only the valid points
    return projected_points[:valid_count].copy()

def get_mesh_name(mesh, idx):
    try:
        return f'{mesh.source.file_name.lower()}_{idx}'
    except AttributeError:
        return f'{mesh.metadata.get("name", mesh.metadata.get("file_name", f"unknown")).lower()}_{idx}'

def get_time_str():
    """
    Return current time in the format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
%%writefile transform_utils.py
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""
Utility functions of matrix and vector transformations.

Portions of this file are adapted from OmniGibson:
- Upstream: https://github.com/StanfordVL/OmniGibson
- Upstream license: MIT

NOTE: convention for quaternions is (x, y, z, w)
"""

import math
import numpy as np
from numba import njit
from scipy.spatial.transform import Rotation as R

PI = np.pi
EPS = np.finfo(float).eps * 4.0

# axis sequences for Euler angles
_NEXT_AXIS = [1, 2, 0, 1]

# map axes strings to/from tuples of inner axis, parity, repetition, frame
_AXES2TUPLE = {
    "sxyz": (0, 0, 0, 0), "sxyx": (0, 0, 1, 0), "sxzy": (0, 1, 0, 0), "sxzx": (0, 1, 1, 0),
    "syzx": (1, 0, 0, 0), "syzy": (1, 0, 1, 0), "syxz": (1, 1, 0, 0), "syxy": (1, 1, 1, 0),
    "szxy": (2, 0, 0, 0), "szxz": (2, 0, 1, 0), "szyx": (2, 1, 0, 0), "szyz": (2, 1, 1, 0),
    "rzyx": (0, 0, 0, 1), "rxyx": (0, 0, 1, 1), "ryzx": (0, 1, 0, 1), "rxzx": (0, 1, 1, 1),
    "rxzy": (1, 0, 0, 1), "ryzy": (1, 0, 1, 1), "rzxy": (1, 1, 0, 1), "ryxy": (1, 1, 1, 1),
    "ryxz": (2, 0, 0, 1), "rzxz": (2, 0, 1, 1), "rxyz": (2, 1, 0, 1), "rzyz": (2, 1, 1, 1),
}
_TUPLE2AXES = dict((v, k) for k, v in _AXES2TUPLE.items())

@njit(cache=True, fastmath=True)
def _mat2quat_single(R):
    t = R[0, 0] + R[1, 1] + R[2, 2]
    if t > 0.0:
        s = 0.5 / np.sqrt(t + 1.0)
        w = 0.25 / s
        x = (R[2, 1] - R[1, 2]) * s
        y = (R[0, 2] - R[2, 0]) * s
        z = (R[1, 0] - R[0, 1]) * s
    else:
        if R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
            s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
            w = (R[2, 1] - R[1, 2]) / s; x = 0.25 * s
            y = (R[0, 1] + R[1, 0]) / s; z = (R[0, 2] + R[2, 0]) / s
        elif R[1, 1] > R[2, 2]:
            s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
            w = (R[0, 2] - R[2, 0]) / s; x = (R[0, 1] + R[1, 0]) / s
            y = 0.25 * s; z = (R[1, 2] + R[2, 1]) / s
        else:
            s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
            w = (R[1, 0] - R[0, 1]) / s; x = (R[0, 2] + R[2, 0]) / s
            y = (R[1, 2] + R[2, 1]) / s; z = 0.25 * s
    return np.array((x, y, z, w), dtype=R.dtype)

@njit(cache=True, fastmath=True)
def _quat2mat_single(q):
    x, y, z, w = q
    xx, yy, zz = x*x, y*y, z*z
    xy, xz, yz = x*y, x*z, y*z
    wx, wy, wz = w*x, w*y, w*z
    M = np.empty((3, 3), dtype=q.dtype)
    M[0,0]=1.0-2.0*(yy+zz); M[0,1]=2.0*(xy-wz); M[0,2]=2.0*(xz+wy)
    M[1,0]=2.0*(xy+wz); M[1,1]=1.0-2.0*(xx+zz); M[1,2]=2.0*(yz-wx)
    M[2,0]=2.0*(xz-wy); M[2,1]=2.0*(yz+wx); M[2,2]=1.0-2.0*(xx+yy)
    return M

@njit(cache=True, fastmath=True)
def _mat2quat_kernel(poses_mat, out):
    N = poses_mat.shape[0]
    for i in range(N):
        out[i, :3] = poses_mat[i, :3, 3]
        out[i, 3:] = _mat2quat_single(poses_mat[i, :3, :3])

@njit(cache=True, fastmath=True)
def _quat2mat_kernel(poses_quat, out):
    N = poses_quat.shape[0]
    for i in range(N):
        out[i, :3, 3] = poses_quat[i, :3]
        R = _quat2mat_single(poses_quat[i, 3:])
        out[i, 0, :3] = R[0]; out[i, 1, :3] = R[1]; out[i, 2, :3] = R[2]

def convert_pose_mat2quat(poses_mat):
    was_single = poses_mat.ndim == 2
    if was_single: poses_mat = poses_mat[None]
    out = np.empty((poses_mat.shape[0], 7), dtype=poses_mat.dtype)
    _mat2quat_kernel(poses_mat, out)
    return out[0] if was_single else out

def convert_pose_quat2mat(poses_quat):
    was_single = poses_quat.ndim == 1
    if was_single: poses_quat = poses_quat[None]
    out = np.empty((poses_quat.shape[0], 4, 4), dtype=poses_quat.dtype)
    out[:, 3, :] = np.array((0., 0., 0., 1.), dtype=poses_quat.dtype)
    _quat2mat_kernel(poses_quat, out)
    return out[0] if was_single else out

def mat2quat(rmat):
    return R.from_matrix(rmat).as_quat()

def quat2mat(quaternion):
    return R.from_quat(quaternion).as_matrix()

def euler2mat(euler):
    euler = np.asarray(euler, dtype=np.float64)
    assert euler.shape[-1] == 3
    return R.from_euler("xyz", euler).as_matrix()

def mat2euler(rmat):
    M = np.array(rmat, dtype=rmat.dtype, copy=False)[:3, :3]
    return R.from_matrix(M).as_euler("xyz")

def euler2quat(euler):
    return R.from_euler("xyz", euler).as_quat()

def quat2euler(quat):
    return R.from_quat(quat).as_euler("xyz")

def quat2axisangle(quat):
    return R.from_quat(quat).as_rotvec()

def axisangle2quat(vec):
    return R.from_rotvec(vec).as_quat()

def mat2pose(hmat):
    pos = hmat[:3, 3]
    orn = mat2quat(hmat[:3, :3])
    return pos, orn

def pose2mat(pose):
    homo_pose_mat = np.zeros((4, 4), dtype=pose[0].dtype)
    homo_pose_mat[:3, :3] = quat2mat(pose[1])
    homo_pose_mat[:3, 3] = np.array(pose[0], dtype=pose[0].dtype)
    homo_pose_mat[3, 3] = 1.0
    return homo_pose_mat

def make_pose(translation, rotation):
    pose = np.zeros((4, 4))
    pose[:3, :3] = rotation
    pose[:3, 3] = translation
    pose[3, 3] = 1.0
    return pose

def pose_inv(pose_mat):
    pose_inv = np.zeros((4, 4))
    pose_inv[:3, :3] = pose_mat[:3, :3].T
    pose_inv[:3, 3] = -pose_inv[:3, :3].dot(pose_mat[:3, 3])
    pose_inv[3, 3] = 1.0
    return pose_inv

def convert_pose_euler2mat(poses_euler):
    batched = poses_euler.ndim == 2
    if not batched: poses_euler = poses_euler[None]
    poses_mat = np.tile(np.eye(4), (len(poses_euler), 1, 1))
    poses_mat[:, :3, 3] = poses_euler[:, :3]
    for i in range(len(poses_euler)):
        poses_mat[i, :3, :3] = euler2mat(poses_euler[i, 3:])
    if not batched: poses_mat = poses_mat[0]
    return poses_mat

def convert_pose_euler2quat(poses_euler):
    batched = poses_euler.ndim == 2
    if not batched: poses_euler = poses_euler[None]
    poses_quat = np.empty((len(poses_euler), 7))
    poses_quat[:, :3] = poses_euler[:, :3]
    for i in range(len(poses_euler)):
        poses_quat[i, 3:] = euler2quat(poses_euler[i, 3:])
    if not batched: poses_quat = poses_quat[0]
    return poses_quat

def convert_pose_quat2euler(poses_quat):
    batched = poses_quat.ndim == 2
    if not batched: poses_quat = poses_quat[None]
    poses_euler = np.empty((len(poses_quat), 6))
    poses_euler[:, :3] = poses_quat[:, :3]
    for i in range(len(poses_quat)):
        poses_euler[i, 3:] = quat2euler(poses_quat[i, 3:])
    if not batched: poses_euler = poses_euler[0]
    return poses_euler

def unit_vector(data, axis=None, out=None):
    if out is None:
        data = np.array(data, dtype=data.dtype, copy=True)
        if data.ndim == 1:
            data /= math.sqrt(np.dot(data, data))
            return data
    else:
        if out is not data:
            out[:] = np.array(data, copy=False)
        data = out
    length = np.atleast_1d(np.sum(data * data, axis))
    np.sqrt(length, length)
    if axis is not None:
        length = np.expand_dims(length, axis)
    data /= length
    if out is None:
        return data

def normalize(v, axis=None, eps=1e-10):
    norm = np.linalg.norm(v, axis=axis, keepdims=True)
    return v / np.where(norm < eps, eps, norm)

def anorm(x, axis=None, keepdims=False):
    return np.linalg.norm(x, axis=axis, keepdims=keepdims)

def quat_multiply(quaternion1, quaternion0):
    x0,y0,z0,w0 = quaternion0; x1,y1,z1,w1 = quaternion1
    return np.array((
        x1*w0+y1*z0-z1*y0+w1*x0, -x1*z0+y1*w0+z1*x0+w1*y0,
        x1*y0-y1*x0+z1*w0+w1*z0, -x1*x0-y1*y0-z1*z0+w1*w0,
    ), dtype=quaternion0.dtype)

def quat_conjugate(quaternion):
    return np.array((-quaternion[0],-quaternion[1],-quaternion[2],quaternion[3]), dtype=quaternion.dtype)

def quat_inverse(quaternion):
    return quat_conjugate(quaternion) / np.dot(quaternion, quaternion)

def quat_distance(quaternion1, quaternion0):
    return quat_multiply(quaternion1, quat_inverse(quaternion0))

def quat_slerp(quat0, quat1, fraction, shortestpath=True):
    q0 = unit_vector(quat0[:4]); q1 = unit_vector(quat1[:4])
    if fraction == 0.0: return q0
    elif fraction == 1.0: return q1
    d = np.dot(q0, q1)
    if abs(abs(d)-1.0) < EPS: return q0
    if shortestpath and d < 0.0: d=-d; q1*=-1.0
    angle = math.acos(np.clip(d,-1,1))
    if abs(angle) < EPS: return q0
    isin = 1.0/math.sin(angle)
    q0 *= math.sin((1.0-fraction)*angle)*isin
    q1 *= math.sin(fraction*angle)*isin
    q0 += q1
    return q0

@njit(cache=True, fastmath=True)
def quat_slerp_jitted(quat0, quat1, fraction, shortestpath=True):
    EPS = 1e-8
    q0 = quat0/np.linalg.norm(quat0); q1 = quat1/np.linalg.norm(quat1)
    if fraction == 0.0: return q0
    elif fraction == 1.0: return q1
    d = np.dot(q0,q1)
    if np.abs(np.abs(d)-1.0) < EPS: return q0
    if shortestpath and d < 0.0: d=-d; q1*=-1.0
    if d < -1.0: d=-1.0
    elif d > 1.0: d=1.0
    angle = np.arccos(d)
    if np.abs(angle) < EPS: return q0
    isin = 1.0/np.sin(angle)
    q0 *= np.sin((1.0-fraction)*angle)*isin
    q1 *= np.sin(fraction*angle)*isin
    q0 += q1
    return q0

def rotation_translation_from_4x4(T):
    return T[:3,:3], T[:3,3]

def average_poses(poses_4x4):
    if len(poses_4x4) == 1: return poses_4x4[0]
    R_sum = np.zeros((3,3), dtype=np.float64)
    t_sum = np.zeros((3,), dtype=np.float64)
    for p in poses_4x4:
        Rp,tp = rotation_translation_from_4x4(p)
        R_sum += Rp; t_sum += tp
    R_sum /= len(poses_4x4); t_sum /= len(poses_4x4)
    U,S,Vt = np.linalg.svd(R_sum)
    R_ortho = U@Vt
    if np.linalg.det(R_ortho) < 0: R_ortho[:,-1] *= -1
    out = np.eye(4); out[:3,:3]=R_ortho; out[:3,3]=t_sum
    return out

def rotation_error_deg(R1, R2):
    if np.linalg.det(R1)<0: R1=-R1
    if np.linalg.det(R2)<0: R2=-R2
    dot_product = np.clip(np.trace(R1.T@R2), -3.0, 3.0)
    return np.degrees(np.arccos((dot_product-1)/2))

def translation_error(t1, t2):
    return np.linalg.norm(t1-t2)

def cartesian_to_polar(x, y):
    rho = np.sqrt(x**2+y**2); phi = np.arctan2(y,x)
    return rho, phi

def deg2rad(deg): return deg*np.pi/180.
def rad2deg(rad): return rad*180./np.pi

def l2_distance(v1,v2): return np.linalg.norm(np.array(v1)-np.array(v2))

def matrix_inverse(matrix): return np.linalg.inv(matrix)

def pose_in_A_to_pose_in_B(pose_A, pose_A_in_B): return pose_A_in_B.dot(pose_A)

def vec(values): return np.array(values, dtype=np.float32)
def mat4(array): return np.array(array, dtype=np.float32).reshape((4,4))

def vec2quat(vec, up=(0,0,1.0)):
    vec_n=vec/np.linalg.norm(vec); up_n=np.array(up)/np.linalg.norm(up)
    s_n=np.cross(up_n,vec_n); u_n=np.cross(vec_n,s_n)
    return mat2quat(np.array([vec_n,s_n,u_n]).T)

def vecs2axisangle(vec0,vec1):
    vec0=normalize(vec0,axis=-1); vec1=normalize(vec1,axis=-1)
    return np.cross(vec0,vec1)*np.arccos((vec0*vec1).sum(-1,keepdims=True))

def vecs2quat(vec0,vec1,normalized=False):
    if not normalized: vec0=normalize(vec0,axis=-1); vec1=normalize(vec1,axis=-1)
    cos_theta=np.sum(vec0*vec1,axis=-1,keepdims=True)
    q_un=np.where(cos_theta==-1,np.array([1.,0,0,0]),np.concatenate([np.cross(vec0,vec1),1+cos_theta],axis=-1))
    return q_un/np.linalg.norm(q_un,axis=-1,keepdims=True)

def ewma_vectorized(data, alpha, offset=None, dtype=None, order="C", out=None):
    data=np.array(data,copy=False)
    if dtype is None:
        dtype=np.float32 if data.dtype==np.float32 else np.float64
    else: dtype=np.dtype(dtype)
    if data.ndim>1: data=data.reshape(-1,order)
    if out is None: out=np.empty_like(data,dtype=dtype)
    else: assert out.shape==data.shape and out.dtype==dtype
    if data.size<1: return out
    if offset is None: offset=data[0]
    alpha=np.array(alpha,copy=False).astype(dtype,copy=False)
    scaling_factors=np.power(1.0-alpha,np.arange(data.size+1,dtype=dtype),dtype=dtype)
    np.multiply(data,(alpha*scaling_factors[-2])/scaling_factors[:-1],dtype=dtype,out=out)
    np.cumsum(out,dtype=dtype,out=out)
    out/=scaling_factors[-2::-1]
    if offset != 0:
        offset=np.array(offset,copy=False).astype(dtype,copy=False)
        out+=offset*scaling_factors[1:]
    return out

def check_quat_right_angle(quat, atol=5e-2):
    return np.any(np.isclose(np.abs(quat).sum(),np.array([1.0,1.414,2.0]),atol=atol))

def z_angle_from_quat(quat):
    rotated_X_axis=R.from_quat(quat).apply([1,0,0])
    return np.arctan2(rotated_X_axis[1],rotated_X_axis[0])

def z_rotation_from_quat(quat):
    return R.from_euler("z",z_angle_from_quat(quat)).as_quat()

def get_orientation_error(target_orn, current_orn):
    current_orn=np.array([current_orn[3],current_orn[0],current_orn[1],current_orn[2]])
    target_orn=np.array([target_orn[3],target_orn[0],target_orn[1],target_orn[2]])
    pinv=np.zeros((3,4))
    pinv[0,:]=[-current_orn[1],current_orn[0],-current_orn[3],current_orn[2]]
    pinv[1,:]=[-current_orn[2],current_orn[3],current_orn[0],-current_orn[1]]
    pinv[2,:]=[-current_orn[3],-current_orn[2],current_orn[1],current_orn[0]]
    return 2.0*pinv.dot(np.array(target_orn))

def get_pose_error(target_pose, current_pose):
    error=np.zeros(6)
    pos_err=target_pose[:3,3]-current_pose[:3,3]
    r1,r2,r3=current_pose[:3,0],current_pose[:3,1],current_pose[:3,2]
    r1d,r2d,r3d=target_pose[:3,0],target_pose[:3,1],target_pose[:3,2]
    rot_err=0.5*(np.cross(r1,r1d)+np.cross(r2,r2d)+np.cross(r3,r3d))
    error[:3]=pos_err; error[3:]=rot_err
    return error

def random_quat(rand=None):
    if rand is None: rand=np.random.rand(3)
    else: assert len(rand)==3
    r1=np.sqrt(1.0-rand[0]); r2=np.sqrt(rand[0])
    pi2=math.pi*2.0; t1=pi2*rand[1]; t2=pi2*rand[2]
    return np.array((np.sin(t1)*r1,np.cos(t1)*r1,np.sin(t2)*r2,np.cos(t2)*r2),dtype=np.float32)

def clip_translation(dpos,limit):
    input_norm=np.linalg.norm(dpos)
    return (dpos*limit/input_norm,True) if input_norm>limit else (dpos,False)

def clip_rotation(quat,limit):
    clipped=False; quat=quat/np.linalg.norm(quat)
    den=np.sqrt(max(1-quat[3]*quat[3],0))
    if den==0: return quat,clipped
    x,y,z=quat[0]/den,quat[1]/den,quat[2]/den
    a=2*math.acos(quat[3])
    if abs(a)>limit:
        a=limit*np.sign(a)/2; sa=math.sin(a); ca=math.cos(a)
        quat=np.array([x*sa,y*sa,z*sa,ca]); clipped=True
    return quat,clipped

def rotation_matrix(angle,direction,point=None):
    sina=math.sin(angle); cosa=math.cos(angle)
    direction=unit_vector(direction[:3])
    Rm=np.array(((cosa,0.,0.),(0.,cosa,0.),(0.,0.,cosa)),dtype=direction.dtype)
    Rm+=np.outer(direction,direction)*(1.0-cosa)
    direction*=sina
    Rm+=np.array(((0.,-direction[2],direction[1]),(direction[2],0.,-direction[0]),(-direction[1],direction[0],0.)),dtype=direction.dtype)
    M=np.identity(4); M[:3,:3]=Rm
    if point is not None:
        point=np.array(point[:3],dtype=direction.dtype,copy=False)
        M[:3,3]=point-np.dot(Rm,point)
    return M

def frustum(left,right,bottom,top,znear,zfar):
    assert right!=left and bottom!=top and znear!=zfar
    M=np.zeros((4,4),dtype=np.float32)
    M[0,0]=+2.*znear/(right-left); M[2,0]=(right+left)/(right-left)
    M[1,1]=+2.*znear/(top-bottom); M[2,1]=(top+bottom)/(top-bottom)
    M[2,2]=-(zfar+znear)/(zfar-znear); M[3,2]=-2.*znear*zfar/(zfar-znear)
    M[2,3]=-1.; return M

def ortho(left,right,bottom,top,znear,zfar):
    assert right!=left and bottom!=top and znear!=zfar
    M=np.zeros((4,4),dtype=np.float32)
    M[0,0]=2./(right-left); M[1,1]=2./(top-bottom); M[2,2]=-2./(zfar-znear)
    M[3,0]=-(right+left)/(right-left); M[3,1]=-(top+bottom)/(top-bottom)
    M[3,2]=-(zfar+znear)/(zfar-znear); M[3,3]=1.; return M

def perspective(fovy,aspect,znear,zfar):
    assert znear!=zfar
    h=np.tan(fovy/360.*np.pi)*znear; w=h*aspect
    return frustum(-w,w,-h,h,znear,zfar)

def _skew_symmetric_translation(pos_A_in_B):
    return np.array([0.,-pos_A_in_B[2],pos_A_in_B[1],
                     pos_A_in_B[2],0.,-pos_A_in_B[0],
                     -pos_A_in_B[1],pos_A_in_B[0],0.]).reshape((3,3))

def vel_in_A_to_vel_in_B(vel_A,ang_vel_A,pose_A_in_B):
    pos_A_in_B=pose_A_in_B[:3,3]; rot_A_in_B=pose_A_in_B[:3,:3]
    skew_symm=_skew_symmetric_translation(pos_A_in_B)
    vel_B=rot_A_in_B.dot(vel_A)+skew_symm.dot(rot_A_in_B.dot(ang_vel_A))
    ang_vel_B=rot_A_in_B.dot(ang_vel_A)
    return vel_B,ang_vel_B

def force_in_A_to_force_in_B(force_A,torque_A,pose_A_in_B):
    pos_A_in_B=pose_A_in_B[:3,3]; rot_A_in_B=pose_A_in_B[:3,:3]
    skew_symm=_skew_symmetric_translation(pos_A_in_B)
    force_B=rot_A_in_B.T.dot(force_A)
    torque_B=-rot_A_in_B.T.dot(skew_symm.dot(force_A))+rot_A_in_B.T.dot(torque_A)
    return force_B,torque_B

def pose_transform(pos1,quat1,pos0,quat0):
    mat0=pose2mat((pos0,quat0)); mat1=pose2mat((pos1,quat1))
    return mat2pose(mat1@mat0)

def invert_pose_transform(pos,quat):
    mat=pose2mat((pos,quat)); return mat2pose(pose_inv(mat))

def relative_pose_transform(pos1,quat1,pos0,quat0):
    mat0=pose2mat((pos0,quat0)); mat1=pose2mat((pos1,quat1))
    return mat2pose(pose_inv(mat0)@mat1)

def get_orientation_diff_in_radian(orn0,orn1):
    vec0=quat2axisangle(orn0); vec0/=np.linalg.norm(vec0)
    vec1=quat2axisangle(orn1); vec1/=np.linalg.norm(vec1)
    return np.arccos(np.dot(vec0,vec1))

def quat_distance_batched(quaternions1,quaternions0):
    dists=np.zeros_like(quaternions1)
    for i in range(quaternions1.shape[0]):
        dists[i]=quat_distance(quaternions1[i],quaternions0[i])
    return dists

def random_axis_angle(angle_limit=None,random_state=None):
    if angle_limit is None: angle_limit=2.*np.pi
    if random_state is not None:
        assert isinstance(random_state,np.random.RandomState); npr=random_state
    else: npr=np.random
    random_axis=npr.randn(3); random_axis/=np.linalg.norm(random_axis)
    random_angle=npr.uniform(low=0.,high=angle_limit)
    return random_axis,random_angle

In [ ]:
%%writefile real/droid_utils.py
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import os
import numpy as np
import sys as _sys
_THIS_DIR = os.path.dirname(os.path.abspath(__file__))
_ROOT_DIR = os.path.normpath(os.path.join(_THIS_DIR, '..'))
if _ROOT_DIR not in _sys.path:
    _sys.path.insert(0, _ROOT_DIR)
import cv2
from tqdm import tqdm
import json
import h5py
import glob
import transform_utils
import tempfile
import shutil
from real.gcs_utils import is_gcs_path, get_local_path, list_gcs_files
try:
    import pyzed.sl as sl
except ImportError as exc:
    sl = None
    _PYZED_IMPORT_ERROR = exc
else:
    _PYZED_IMPORT_ERROR = None


def _require_pyzed():
    if sl is None:
        raise ImportError("pyzed is required for ZED SVO processing") from _PYZED_IMPORT_ERROR


def _binary_search_latest_range(arr, left, right, target):
    if arr[right] <= target or right == left:
        return arr[right]
    mid = ((left + right) >> 1) + 1
    if arr[mid] <= target:
        return _binary_search_latest_range(arr, mid, right, target)
    return _binary_search_latest_range(arr, left, mid - 1, target)


def _binary_search_latest(arr, target):
    if len(arr) <= 0:
        raise ValueError("input array should contain at least one element")
    return _binary_search_latest_range(arr, 0, len(arr) - 1, target)


def _binary_search_closest(arr, target):
    """adapted from rh20t_api"""
    if target in arr:
        return target
    prev_idx = arr.index(_binary_search_latest(arr, target))
    if prev_idx == len(arr) - 1:
        return arr[prev_idx]
    prev_val = arr[prev_idx]
    next_val = arr[prev_idx + 1]
    return prev_val if abs(prev_val - target) < abs(next_val - target) else next_val


def _resolve_svo_path(scene_path, svo_path):
    if not svo_path.startswith('/') and not is_gcs_path(svo_path):
        return os.path.join(scene_path, *svo_path.split('/')[-3:])
    return svo_path


def _init_camera_entries(metadata, include_wrist_cam):
    camera_names = ['wrist', 'ext1', 'ext2'] if include_wrist_cam else ['ext1', 'ext2']
    data_dict = {}
    camera_specs = []
    for camera_name in camera_names:
        serial_key = f'{camera_name}_cam_serial'
        if serial_key not in metadata:
            raise KeyError(f"Camera {camera_name} not found in metadata")
        camera_serial = metadata[serial_key]
        data_dict[camera_serial] = {}
        if camera_name == 'wrist':
            data_dict[camera_serial]['extrinsic'] = np.eye(4)
        else:
            extrinsic_key = f'{camera_name}_cam_extrinsics'
            if extrinsic_key not in metadata:
                raise KeyError(f"Extrinsics for camera {camera_name} not found in metadata")
            extrinsic_6 = np.array(metadata[extrinsic_key])
            extrinsic_4x4 = transform_utils.convert_pose_euler2mat(extrinsic_6[None])[0]
            extrinsic_4x4 = np.linalg.inv(extrinsic_4x4)
            data_dict[camera_serial]['extrinsic'] = extrinsic_4x4
        camera_specs.append((camera_name, camera_serial))
    return data_dict, camera_specs


def _open_svo_camera(local_svo_path, include_depth):
    init_params = sl.InitParameters()
    init_params.set_from_svo_file(local_svo_path)
    init_params.svo_real_time_mode = False
    if include_depth:
        init_params.depth_mode = sl.DEPTH_MODE.ULTRA
    else:
        init_params.depth_mode = sl.DEPTH_MODE.NONE
    init_params.coordinate_units = sl.UNIT.METER
    zed = sl.Camera()
    err = zed.open(init_params)
    if err != sl.ERROR_CODE.SUCCESS:
        raise ValueError(f"Error opening SVO file {local_svo_path}: {err}")
    camera_info = zed.get_camera_information()
    return zed, camera_info


def _extract_intrinsics_and_baseline(calibration_params, downscale_ratio):
    fx = calibration_params.left_cam.fx
    fy = calibration_params.left_cam.fy
    cx = calibration_params.left_cam.cx
    cy = calibration_params.left_cam.cy
    intrinsic = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]])
    baseline = calibration_params.stereo_transform.get_translation().get()[0]
    if downscale_ratio != 1.0:
        intrinsic[0,0]*=downscale_ratio; intrinsic[1,1]*=downscale_ratio
        intrinsic[0,2]*=downscale_ratio; intrinsic[1,2]*=downscale_ratio
    return intrinsic, baseline


def _extract_svo_frames(zed, camera_name, include_stereo, include_depth, downscale_ratio, max_frames):
    rgb_frames = []
    right_frames = [] if include_stereo else None
    depth_frames = [] if include_depth else None
    left_image = sl.Mat()
    right_image = sl.Mat() if include_stereo else None
    depth_image = sl.Mat() if include_depth else None
    runtime_params = sl.RuntimeParameters()
    nb_frames = zed.get_svo_number_of_frames()
    if max_frames > 0:
        nb_frames = min(nb_frames, max_frames)
    print(f"Extracting {nb_frames} frames from SVO file for camera {camera_name}...")
    timestamps = []
    frame_count = 0
    with tqdm(total=nb_frames, desc=f"Processing {camera_name} camera") as pbar:
        while frame_count < nb_frames:
            err = zed.grab(runtime_params)
            if err == sl.ERROR_CODE.SUCCESS:
                zed.retrieve_image(left_image, sl.VIEW.LEFT)
                rgb = left_image.get_data().copy()
                if rgb.shape[2] == 4:
                    rgb = cv2.cvtColor(rgb, cv2.COLOR_BGRA2RGB)
                if downscale_ratio != 1.0:
                    rgb = cv2.resize(rgb, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_LINEAR)
                rgb_frames.append(rgb)
                ts = zed.get_timestamp(sl.TIME_REFERENCE.IMAGE).get_milliseconds()
                timestamps.append(ts)
                if include_stereo:
                    zed.retrieve_image(right_image, sl.VIEW.RIGHT)
                    right = right_image.get_data().copy()
                    if right.shape[2] == 4:
                        right = cv2.cvtColor(right, cv2.COLOR_BGRA2RGB)
                    if downscale_ratio != 1.0:
                        right = cv2.resize(right, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_LINEAR)
                    right_frames.append(right)
                if include_depth:
                    zed.retrieve_measure(depth_image, sl.MEASURE.DEPTH)
                    depth = depth_image.get_data().copy()
                    if downscale_ratio != 1.0:
                        depth = cv2.resize(depth, None, fx=downscale_ratio, fy=downscale_ratio, interpolation=cv2.INTER_NEAREST)
                    depth_frames.append(depth)
                frame_count += 1; pbar.update(1)
            elif err == sl.ERROR_CODE.END_OF_SVOFILE_REACHED:
                print(f"End of SVO file reached for camera {camera_name}"); break
            else:
                raise ValueError(f"Error grabbing frame from SVO for camera {camera_name}: {err}")
    if not rgb_frames:
        raise ValueError(f"No frames extracted for camera {camera_name}")
    payload = {'rgb': np.stack(rgb_frames, axis=0), 'timestamps': np.array(timestamps)}
    if include_stereo:
        payload['right_frames'] = np.stack(right_frames, axis=0)
    if include_depth:
        payload['depth'] = np.stack(depth_frames, axis=0)
        payload['depth'] = np.nan_to_num(payload['depth'], nan=0.0, posinf=0.0, neginf=0.0, copy=False)
    return payload


def filter_by_timestamps(data, canonical_timestamps, scene_path, is_proprio=False, verbose=False):
    filtered_data = {}
    if is_proprio:
        temp_dir = None
        try:
            if is_gcs_path(scene_path):
                temp_dir = tempfile.mkdtemp()
            trajectory_path = os.path.join(scene_path, "trajectory.h5")
            local_trajectory_path = get_local_path(trajectory_path, temp_dir)
            with h5py.File(local_trajectory_path, 'r') as f:
                proprio_timestamps = np.array(f['observation']['timestamp']['robot_state']['read_end'])
        finally:
            if temp_dir and os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
        proprio_timestamps_list = proprio_timestamps.tolist()
        filtered_data = {}
        modality_errors = {}
        for modality in data:
            if modality in ['pre_sampled_points']:
                filtered_data[modality] = data[modality]
            else:
                assert len(data[modality]) == len(proprio_timestamps_list)
                aligned_data = []
                errors = []
                for target_ts in canonical_timestamps:
                    closest_ts = _binary_search_closest(proprio_timestamps_list, target_ts)
                    idx = proprio_timestamps_list.index(closest_ts)
                    aligned_data.append(data[modality][idx])
                    if verbose: errors.append(abs(target_ts - closest_ts))
                filtered_data[modality] = np.array(aligned_data)
                if verbose and len(errors) > 0:
                    modality_errors[modality] = np.array(errors) / 1000.0
    else:
        modality_errors = {}
        nontemporal_keys = ['intrinsic', 'extrinsic', 'baseline']
        for camera_serial in data:
            filtered_data[camera_serial] = {}
            for key in nontemporal_keys:
                if key in data[camera_serial]:
                    filtered_data[camera_serial][key] = data[camera_serial][key]
            camera_timestamps_list = data[camera_serial]['timestamps'].tolist()
            for key in data[camera_serial]:
                if key in nontemporal_keys: continue
                assert len(data[camera_serial][key]) == len(camera_timestamps_list)
                aligned_data = []
                errors = []
                for target_ts in canonical_timestamps:
                    closest_ts = _binary_search_closest(camera_timestamps_list, target_ts)
                    idx = camera_timestamps_list.index(closest_ts)
                    aligned_data.append(data[camera_serial][key][idx])
                    if verbose: errors.append(abs(target_ts - closest_ts))
                filtered_data[camera_serial][key] = np.array(aligned_data)
                if verbose and len(errors) > 0:
                    modality_errors[f"{camera_serial}_{key}"] = np.array(errors) / 1000.0
    if verbose:
        for modality, errors in modality_errors.items():
            print(f"{modality} alignment errors (seconds):")
            print(f"  Average: {np.mean(errors):.3f}")
            print(f"  Max: {np.max(errors):.3f}")
            print(f"  Min: {np.min(errors):.3f}")
    return filtered_data

def gather_data_dict(scene_path, downscale_ratio=1.0, include_stereo=False, include_depth=True, include_wrist_cam=False, max_frames=-1):
    _require_pyzed()
    data_dict = {}
    temp_dir = None
    try:
        if is_gcs_path(scene_path):
            temp_dir = tempfile.mkdtemp()
        metadata = get_metadata(scene_path)
        data_dict, camera_specs = _init_camera_entries(metadata, include_wrist_cam)
        for camera_name, camera_serial in camera_specs:
            if f'{camera_name}_svo_path' not in metadata:
                raise KeyError(f"SVO path for camera {camera_name} not found in metadata")
            svo_path = metadata[f'{camera_name}_svo_path']
            resolved_svo_path = _resolve_svo_path(scene_path, svo_path)
            local_svo_path = get_local_path(resolved_svo_path, temp_dir)
            print(f"Processing SVO file for camera {camera_name}: {local_svo_path}")
            zed, camera_info = _open_svo_camera(local_svo_path, include_depth)
            try:
                calibration_params = camera_info.camera_configuration.calibration_parameters
                intrinsic, baseline = _extract_intrinsics_and_baseline(calibration_params, downscale_ratio)
                data_dict[camera_serial]['intrinsic'] = intrinsic
                if include_stereo:
                    data_dict[camera_serial]['baseline'] = baseline
                width = camera_info.camera_configuration.resolution.width
                height = camera_info.camera_configuration.resolution.height
                print(f"Camera {camera_name} resolution: {width}x{height}")
                frames_payload = _extract_svo_frames(zed, camera_name, include_stereo=include_stereo,
                    include_depth=include_depth, downscale_ratio=downscale_ratio, max_frames=max_frames)
                data_dict[camera_serial].update(frames_payload)
            finally:
                zed.close()
            print(f"Extracted {data_dict[camera_serial]['rgb'].shape[0]} frames for camera {camera_serial}")
    finally:
        if temp_dir and os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
    return data_dict

def gather_trajectory(scene_path, max_frames=-1):
    temp_dir = None
    try:
        if is_gcs_path(scene_path):
            temp_dir = tempfile.mkdtemp()
        trajectory_path = os.path.join(scene_path, "trajectory.h5")
        local_trajectory_path = get_local_path(trajectory_path, temp_dir)
        proprio_dict = {}
        with h5py.File(local_trajectory_path, 'r') as f:
            joint_positions = np.array(f['observation']['robot_state']['joint_positions'])
            joint_velocities = np.array(f['observation']['robot_state']['joint_velocities'])
            joint_torques = np.array(f['observation']['robot_state']['joint_torques_computed'])
            gripper_positions = np.array(f['observation']['robot_state']['gripper_position'])
            gripper_pose_6 = np.array(f['observation']['robot_state']['cartesian_position'])
            gripper_pose_7 = transform_utils.convert_pose_euler2quat(gripper_pose_6)
            proprio_timestamps = np.array(f['observation']['timestamp']['robot_state']['read_end'])
            if max_frames > 0:
                joint_positions=joint_positions[:max_frames]; joint_velocities=joint_velocities[:max_frames]
                joint_torques=joint_torques[:max_frames]; gripper_positions=gripper_positions[:max_frames]
                gripper_pose_7=gripper_pose_7[:max_frames]; proprio_timestamps=proprio_timestamps[:max_frames]
            proprio_dict['joint_positions']=joint_positions; proprio_dict['joint_velocities']=joint_velocities
            proprio_dict['joint_torques']=joint_torques
            proprio_dict['gripper_positions']=gripper_positions*0.725
            proprio_dict['gripper_pose']=gripper_pose_7; proprio_dict['timestamps']=proprio_timestamps
    finally:
        if temp_dir and os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
    return proprio_dict

def get_uuid(scene_path):
    if is_gcs_path(scene_path):
        metadata_files = list_gcs_files(scene_path, "metadata_*.json")
        if not metadata_files:
            raise FileNotFoundError(f"No metadata files found in {scene_path}")
        metadata_filename = os.path.basename(metadata_files[0])
        if not metadata_filename.startswith("metadata_") or not metadata_filename.endswith(".json"):
            raise ValueError(f"Unexpected metadata filename format: {metadata_filename}")
        return metadata_filename[9:-5]
    metadata_files = glob.glob(os.path.join(scene_path, "metadata_*.json"))
    if not metadata_files:
        raise FileNotFoundError(f"No metadata files found in {scene_path}")
    metadata_filename = os.path.basename(metadata_files[0])
    if not metadata_filename.startswith("metadata_") or not metadata_filename.endswith(".json"):
        raise ValueError(f"Unexpected metadata filename format: {metadata_filename}")
    return metadata_filename[9:-5]

def get_metadata(scene_path):
    temp_dir = None
    try:
        if is_gcs_path(scene_path):
            temp_dir = tempfile.mkdtemp()
            metadata_files = list_gcs_files(scene_path, "metadata_*.json")
            if not metadata_files:
                raise FileNotFoundError(f"No metadata files found in {scene_path}")
            local_metadata_path = get_local_path(metadata_files[0], temp_dir)
            with open(local_metadata_path, 'r') as f:
                metadata = json.load(f)
        else:
            metadata_files = glob.glob(os.path.join(scene_path, "metadata_*.json"))
            if not metadata_files:
                raise FileNotFoundError(f"No metadata files found in {scene_path}")
            with open(metadata_files[0], 'r') as f:
                metadata = json.load(f)
        return metadata
    finally:
        if temp_dir and os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

def load_robot_transforms(transforms_file):
    robot_transforms = {}
    if os.path.exists(transforms_file):
        with open(transforms_file, 'r') as f:
            transforms_data = json.load(f)
        for robot_serial, data in transforms_data.items():
            robot_transforms[robot_serial] = np.array(data['mean_mat'])
        print(f"Loaded gripper2wrist transforms for {len(robot_transforms)} robots.")
    else:
        raise FileNotFoundError(f"Transform file not found: {transforms_file}")
    return robot_transforms

def get_robot_serial(scene_path):
    metadata = get_metadata(scene_path)
    return metadata['robot_serial']

In [ ]:
%%writefile real/extrinsics_io.py
#!/usr/bin/env python3

# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""I/O helpers for extrinsics pipeline outputs."""

from __future__ import annotations

import json
import os
import time
from typing import Callable

import h5py
import numpy as np


def load_precomputed_depth(output_dir: str,
                           uuid: str,
                           data_dict: dict,
                           wrist_serial: str,
                           log_fn: Callable[[str], None]) -> None:
    """Load precomputed FoundationStereo depth from H5 into data_dict."""
    start = time.time()
    h5_path = os.path.join(output_dir, "depth", f"{uuid}_depth.h5")
    if not os.path.exists(h5_path):
        raise FileNotFoundError(f"Precomputed depth file not found: {h5_path}")
    with h5py.File(h5_path, "r") as f:
        if "metadata" not in f:
            raise ValueError(f"Invalid depth file: missing metadata in {h5_path}")
        metadata = f["metadata"]
        if "write_complete" not in metadata.attrs:
            raise ValueError(f"Depth file missing write_complete flag: {h5_path}")
        if not metadata.attrs["write_complete"]:
            raise ValueError(f"Depth file not complete: {h5_path}")
        camera_groups = [key for key in f.keys() if key != "metadata"]
        for camera_group_name in camera_groups:
            camera_serial, _camera_type = camera_group_name.split("+")
            if camera_serial not in data_dict:
                raise ValueError(f"Camera {camera_serial} in depth file but not in data_dict")
            camera_group = f[camera_group_name]
            depth_uint16 = camera_group["depth"][:]  # [T, H, W]
            depth_timestamps = camera_group["timestamps"][:]  # [T]
            depth_meters = depth_uint16.astype(np.float32) / 1000.0
            canonical_timestamps = data_dict[camera_serial]["timestamps"]
            aligned_depth_frames = []
            for canonical_ts in canonical_timestamps:
                time_diffs = np.abs(depth_timestamps - canonical_ts)
                closest_idx = np.argmin(time_diffs)
                if time_diffs[closest_idx] > 50:
                    log_fn(f"Warning: Large timestamp difference ({time_diffs[closest_idx]:.1f}ms) for camera {camera_serial}")
                aligned_depth_frames.append(depth_meters[closest_idx])
            aligned_depth = np.stack(aligned_depth_frames, axis=0)
            data_dict[camera_serial]["stereo_depth"] = aligned_depth
            data_dict[camera_serial]["stereo_intrinsics"] = data_dict[camera_serial]["measured_intrinsics"]
            log_fn(f"Loaded depth for camera {camera_serial}: {aligned_depth.shape}, range: {aligned_depth.min():.3f}-{aligned_depth.max():.3f}m")
    num_cams = len([k for k in data_dict.keys() if k != wrist_serial])
    log_fn(f"Successfully loaded precomputed depth for {num_cams} cameras (time taken: {time.time() - start:.2f}s)")


def write_camera_results(output_dir: str, uuid: str, scene_path: str, data_dict: dict,
                         wrist_serial: str, optimization_metrics: dict | None,
                         error_info: dict | None, log_fn: Callable[[str], None]) -> str:
    """Write camera JSON results to output_dir/cameras."""
    cameras_dir = os.path.join(output_dir, "cameras")
    os.makedirs(cameras_dir, exist_ok=True)
    results = {"uuid": uuid, "scene_path": scene_path}
    if error_info is not None:
        results["error_info"] = error_info
    if optimization_metrics:
        results["optimization_summary"] = optimization_metrics
    for camera_serial in data_dict:
        if camera_serial == wrist_serial: continue
        camera_data = data_dict[camera_serial]
        camera_result = {}
        if "vggt_extrinsics" in camera_data:
            camera_result["vggt_extrinsics"] = camera_data["vggt_extrinsics"].tolist()
        if error_info is None and "optimized_extrinsics" in camera_data:
            camera_result["optimized_extrinsics"] = camera_data["optimized_extrinsics"].tolist()
        if "measured_intrinsics" in camera_data:
            camera_result["measured_intrinsics"] = camera_data["measured_intrinsics"].tolist()
        if "vggt_intrinsics" in camera_data:
            camera_result["vggt_intrinsics"] = camera_data["vggt_intrinsics"].tolist()
        results[camera_serial] = camera_result
    json_path = os.path.join(cameras_dir, f"{uuid}_cameras.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    log_fn(f"Camera results saved to: {json_path}")
    return json_path

In [ ]:
%%writefile compute_extrinsics_utils.py
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import sys; sys.path.append('..')
import os
import json
import numpy as np
if not hasattr(np, "float"):
    np.float = float  # type: ignore[attr-defined]
from typing import List
from real.real_utils import get_mesh_name
from real.droid_utils import get_uuid, load_robot_transforms
import torch
import urdfpy
import h5py

REAL_DIR = os.path.abspath(os.path.dirname(__file__))
DEFAULT_GRIPPER2WRIST_TRANSFORMS_PATH = os.path.join(
    REAL_DIR,
    "real",
    "gripper2wrist_transforms.json",
)

class RobotVisibilityError(Exception):
    """Raised when robot is not visible in cameras during optimization."""
    def __init__(self, message, error_type="no_robot_visible"):
        self.error_type = error_type
        super().__init__(message)

def check_precomputed_depth_exists(scene_path, output_dir, uuid=None):
    if uuid is None:
        uuid = get_uuid(scene_path)
    h5_path = os.path.join(output_dir, "depth", f"{uuid}_depth.h5")
    if not os.path.exists(h5_path):
        return False, h5_path, f"Precomputed depth file not found: {h5_path}"
    with h5py.File(h5_path, 'r') as f:
        if 'metadata' not in f:
            raise ValueError(f"Invalid depth file: missing metadata in {h5_path}")
        metadata = f['metadata']
        if 'write_complete' not in metadata.attrs:
            raise ValueError(f"Depth file missing write_complete flag: {h5_path}")
        if not metadata.attrs['write_complete']:
            raise ValueError(f"Depth file not complete: {h5_path}")
        camera_groups = [key for key in f.keys() if key != 'metadata']
        if len(camera_groups) == 0:
            raise ValueError(f"No camera groups found in depth file: {h5_path}")
    return True, h5_path, None

def check_camera_results_exist(scene_path, output_dir, uuid=None):
    if uuid is None:
        uuid = get_uuid(scene_path)
    json_path = os.path.join(output_dir, "cameras", f"{uuid}_cameras.json")
    if not os.path.exists(json_path):
        return False, json_path, f"Camera results file not found: {json_path}"
    with open(json_path, 'r') as f:
        data = json.load(f)
    if 'uuid' not in data:
        raise ValueError(f"Invalid camera results file: missing uuid in {json_path}")
    if data['uuid'] != uuid:
        raise ValueError(f"UUID mismatch in camera results file: expected {uuid}, got {data['uuid']}")
    camera_count = 0
    required_fields = ['vggt_extrinsics', 'optimized_extrinsics', 'measured_intrinsics', 'vggt_intrinsics']
    for key, value in data.items():
        if key not in ['uuid', 'optimization_summary'] and isinstance(value, dict):
            camera_count += 1
            for field in required_fields:
                if field not in value:
                    raise ValueError(f"Missing required field '{field}' for camera in {json_path}")
    if camera_count != 2:
        raise ValueError(f"No camera data found in results file: {json_path}")
    return True, json_path, None

def deduplicate_coordinates(xy: torch.Tensor, threshold_pixels: float = 0.5):
    if xy.numel() == 0:
        return [] if xy.ndim == 3 else torch.empty(0, dtype=torch.long)
    q = torch.round(xy / threshold_pixels).to(torch.int32)
    stride = 131_071
    code = q[..., 0] * stride + q[..., 1]
    if xy.ndim == 2:
        idx = torch.arange(code.shape[0], device=xy.device)
        uniq, inv = torch.unique(code, return_inverse=True, sorted=False)
        first = torch.full_like(uniq, fill_value=code.shape[0], dtype=torch.long)
        first.scatter_reduce_(0, inv, idx, reduce="amin")
        return first
    else:
        B, N = code.shape[:2]
        batch_idx = torch.arange(N, device=xy.device).expand(B, N)
        out: List[torch.Tensor] = []
        for b in range(B):
            uniq, inv = torch.unique(code[b], return_inverse=True, sorted=False)
            first = torch.full_like(uniq, fill_value=N, dtype=torch.long)
            first.scatter_reduce_(0, inv, batch_idx[b], reduce="amin")
            out.append(first)
        return out

def sample_depth_with_grid_sample(depth: torch.Tensor, xy: torch.Tensor, align_corners: bool = True) -> torch.Tensor:
    assert isinstance(depth, torch.Tensor) and isinstance(xy, torch.Tensor)
    assert depth.device == xy.device
    assert depth.ndim in (2, 3)
    assert xy.ndim == depth.ndim
    if depth.ndim == 3:
        B, H, W = depth.shape
    else:
        H, W = depth.shape; B = 1
    if depth.ndim == 2:
        x, y = xy[:, 0], xy[:, 1]
        grid = torch.stack((2*x/(W-1)-1, 2*y/(H-1)-1), dim=1)
        grid = grid.unsqueeze(0).unsqueeze(0)
        depth_in = depth.unsqueeze(0).unsqueeze(0)
        out = torch.nn.functional.grid_sample(depth_in, grid, mode="bilinear", padding_mode="zeros", align_corners=align_corners)
        return out.view(-1)
    else:
        B, N = xy.shape[:2]
        x, y = xy[..., 0], xy[..., 1]
        grid = torch.stack((2*x/(W-1)-1, 2*y/(H-1)-1), dim=2)
        grid = grid.unsqueeze(1)
        depth_in = depth.unsqueeze(1)
        out = torch.nn.functional.grid_sample(depth_in, grid, mode="bilinear", padding_mode="zeros", align_corners=align_corners)
        return out.squeeze(1).squeeze(1)

def pose_6dof_to_matrix(pose_6dof):
    x, y, z, roll, pitch, yaw = pose_6dof
    cos_r, sin_r = torch.cos(roll), torch.sin(roll)
    cos_p, sin_p = torch.cos(pitch), torch.sin(pitch)
    cos_y, sin_y = torch.cos(yaw), torch.sin(yaw)
    R = torch.zeros(3, 3, device=pose_6dof.device, dtype=pose_6dof.dtype)
    R[0,0]=cos_y*cos_p; R[0,1]=cos_y*sin_p*sin_r-sin_y*cos_r; R[0,2]=cos_y*sin_p*cos_r+sin_y*sin_r
    R[1,0]=sin_y*cos_p; R[1,1]=sin_y*sin_p*sin_r+cos_y*cos_r; R[1,2]=sin_y*sin_p*cos_r-cos_y*sin_r
    R[2,0]=-sin_p; R[2,1]=cos_p*sin_r; R[2,2]=cos_p*cos_r
    T = torch.eye(4, device=pose_6dof.device, dtype=pose_6dof.dtype)
    T[:3,:3]=R; T[:3,3]=torch.stack([x,y,z])
    return T

class RobotMeshRenderer:
    """Simplified robot mesh renderer for extrinsics optimization."""
    def __init__(self, urdf_path, device="cuda", total_samples=25000):
        self.device = device; self.urdf_path = urdf_path; self.dtype = torch.float32
        assert os.path.exists(urdf_path), f"URDF file not found: {urdf_path}"
        self.robot_urdf = urdfpy.URDF.load(urdf_path)
        print(f"Loaded URDF from: {urdf_path}")
        self.mesh_points = {}; self.total_samples = total_samples
        self.fk_cache = {}; self.world_points_cache = {}

    def _get_forward_kinematics(self, joint_positions, gripper_position):
        cache_key = tuple(np.round(joint_positions, 4).tolist() + [round(gripper_position, 4)])
        if cache_key in self.fk_cache: return self.fk_cache[cache_key]
        cfg = {'finger_joint': float(gripper_position)}
        for ji in range(7): cfg[f'panda_joint{ji+1}'] = float(joint_positions[ji])
        fk_result = self.robot_urdf.visual_trimesh_fk(cfg=cfg)
        self.fk_cache[cache_key] = fk_result
        return fk_result

    def _sample_mesh_points(self, fk_result):
        mesh_names=[]; mesh_objects=[]; mesh_areas=[]
        for i, mesh in enumerate(fk_result):
            mesh_name = get_mesh_name(mesh, i)
            if mesh.area <= 0: continue
            effective_area = mesh.area
            if 'hand_camera_part' in mesh_name.lower(): effective_area *= 0.000001
            mesh_names.append(mesh_name); mesh_objects.append(mesh); mesh_areas.append(effective_area)
        if not mesh_names: raise ValueError("No meshes with positive area found for robot sampling.")
        total_area = sum(mesh_areas)
        if total_area <= 0: raise ValueError("Total mesh area is non-positive; cannot sample robot points.")
        for name, mesh, area in zip(mesh_names, mesh_objects, mesh_areas):
            ratio = area / total_area
            count = max(200, int(self.total_samples * ratio))
            points_3d = mesh.sample(count)
            self.mesh_points[name] = torch.from_numpy(points_3d.astype(np.float32)).to(device=self.device, dtype=self.dtype)

    def _get_world_points(self, fk_result):
        fk_poses_key = []
        for i, mesh in enumerate(fk_result):
            mesh_name = get_mesh_name(mesh, i)
            if mesh_name in self.mesh_points or not self.mesh_points:
                pose_flat = fk_result[mesh].flatten()
                fk_poses_key.extend(np.round(pose_flat, 4).tolist())
        cache_key = tuple(fk_poses_key)
        if cache_key in self.world_points_cache: return self.world_points_cache[cache_key]
        if not self.mesh_points: self._sample_mesh_points(fk_result)
        fk_poses = {}
        for i, mesh in enumerate(fk_result):
            mesh_name = get_mesh_name(mesh, i)
            if mesh_name in self.mesh_points: fk_poses[mesh_name] = fk_result[mesh]
        all_world_points = []
        for mesh_name, local_points in self.mesh_points.items():
            if mesh_name not in fk_poses: continue
            pose = torch.as_tensor(fk_poses[mesh_name], dtype=self.dtype, device=self.device)
            ones = torch.ones((local_points.shape[0], 1), device=self.device, dtype=self.dtype)
            points_h = torch.cat([local_points, ones], dim=1)
            world_pts = torch.mm(points_h, pose.T)[:, :3]
            all_world_points.append(world_pts)
        assert len(all_world_points) > 0, "No valid robot points found"
        result = torch.cat(all_world_points, dim=0)
        self.world_points_cache[cache_key] = result
        return result

    def resample(self, total_samples=None):
        if total_samples is not None and total_samples > 0: self.total_samples = total_samples
        self.mesh_points.clear(); self.world_points_cache.clear()

def get_robot_transform(robot_serial, transforms_file=DEFAULT_GRIPPER2WRIST_TRANSFORMS_PATH):
    all_transforms = load_robot_transforms(transforms_file)
    if robot_serial not in all_transforms:
        available = list(all_transforms.keys())
        raise ValueError(
            f"Robot serial {robot_serial} not found in transforms file {transforms_file}. "
            f"Available robot serials: {available[:10]}{'...' if len(available) > 10 else ''}"
        )
    return np.array(all_transforms[robot_serial])

In [ ]:
%%writefile real/vggt_forward.py
#!/usr/bin/env python3

# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""VGGT forward-pass wrapper for extrinsics + optional depth/point prediction."""
import os
import sys
import shutil
import tempfile
from contextlib import contextmanager

import numpy as np
import torch
from PIL import Image

REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
VGGT_ROOT = os.path.join(REPO_ROOT, "third_party", "vggt")
if VGGT_ROOT not in sys.path:
    sys.path.append(VGGT_ROOT)

try:
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri
    from vggt.utils.load_fn import load_and_preprocess_images
except ImportError as e:
    print("Error: Could not import VGGT utilities. Ensure the vggt submodule is initialized and installed.")
    raise e


@contextmanager
def stage_vggt_images(images):
    """Write RGB images to a temp dir for VGGT and yield file paths."""
    temp_dir = tempfile.mkdtemp(prefix="vggt_frames_")
    paths = []
    try:
        for idx, img in enumerate(images):
            path = os.path.join(temp_dir, f"frame_{idx:05d}.png")
            Image.fromarray(img).save(path)
            paths.append(path)
        yield paths
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)


class VGGTForwardPass:
    """VGGT forward pass for camera extrinsics/intrinsics."""

    def __init__(self, model, device="cuda"):
        self.model = model
        self.device = device

    @torch.inference_mode()
    @torch.cuda.amp.autocast(enabled=True)
    def __call__(self, image_paths):
        images_tensor = load_and_preprocess_images(image_paths).to(self.device)
        images_tensor = images_tensor[None]  # [batch=1, N, 3, H, W]
        aggregated_tokens_list, _ = self.model.aggregator(images_tensor)
        pose_enc = self.model.camera_head(aggregated_tokens_list)[-1]
        extrinsic_np, intrinsic_np = pose_encoding_to_extri_intri(pose_enc, images_tensor.shape[-2:])
        extrinsic_np = extrinsic_np[0].cpu().numpy()  # [N, 3, 4]
        intrinsic_np = intrinsic_np[0].cpu().numpy()  # [N, 3, 3]
        extrinsic_4x4_list = []
        for i in range(extrinsic_np.shape[0]):
            M = np.eye(4, dtype=np.float32)
            M[:3, :4] = extrinsic_np[i]
            extrinsic_4x4_list.append(M)
        return {
            "extrinsics_3x4": extrinsic_np,
            "extrinsics_4x4": np.stack(extrinsic_4x4_list, axis=0),
            "intrinsics": intrinsic_np,
        }

In [ ]:
%%writefile real/extrinsics_pipeline.py
#!/usr/bin/env python3

# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
"""Extrinsics optimization helpers for DROID scenes."""

import os
import time

import cv2
import numpy as np
import torch

from compute_extrinsics_utils import (
    RobotVisibilityError,
    check_camera_results_exist,
    check_precomputed_depth_exists,
    deduplicate_coordinates,
    get_robot_transform,
    pose_6dof_to_matrix,
    sample_depth_with_grid_sample,
)
from real.droid_utils import (
    filter_by_timestamps,
    gather_data_dict,
    gather_trajectory,
    get_metadata,
    get_robot_serial,
    get_uuid,
)
from real.extrinsics_io import load_precomputed_depth, write_camera_results
from real.real_utils import get_time_str
from real.vggt_forward import stage_vggt_images
import transform_utils

def _print(*args, **kwargs):
    """Helper function that wraps print with automatic time prefixing and flush=True"""
    if args and isinstance(args[0], str) and args[0].startswith(f"[{get_time_str()}]"):
        print(*args, flush=True, **kwargs)
    else:
        if args:
            first_arg = f"[{get_time_str()}] {args[0]}"
            print(first_arg, *args[1:], flush=True, **kwargs)
        else:
            print(f"[{get_time_str()}]", flush=True, **kwargs)

MIN_DEPTH_M = 0.3
MAX_DEPTH_M = 2.0
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
URDF_PATH = os.path.join(
    REPO_ROOT,
    "assets",
    "franka_description",
    "franka_panda_robotiq_2f85_og.urdf",
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ExtrinsicsOptimizer:
    """Optimizes camera extrinsics for a single scene using robot mesh rendering and depth estimation."""
    
    def __init__(self, 
                 scene_path,
                 vggt_forward=None,
                 robot_renderer=None,
                 device="cuda",
                 num_frames=1,
                 downscale_ratio=1.0,
                 min_robot_points=2000):
        self.scene_path = scene_path
        self.vggt_forward = vggt_forward
        self.robot_renderer = robot_renderer
        self.device = device
        self.num_frames = num_frames
        self.downscale_ratio = downscale_ratio
        self.min_robot_points = min_robot_points
        self.data_dict = {}
        self.proprio_dict = {}
        self.meta = get_metadata(self.scene_path)
        self.uuid = self.meta['uuid']
        self.wrist_serial = self.meta["wrist_cam_serial"]
        self.ext_serials = []
        camera_keys = [k for k in self.meta.keys() if k.endswith("_cam_serial")]
        for ck in camera_keys:
            if ck == "wrist_cam_serial":
                continue
            self.ext_serials.append(self.meta[ck])
        self.optimization_metrics = {}
        
    def _load_scene_data(self):
        """Load scene data using gather_data_dict and gather_trajectory with timestamp alignment"""
        start = time.time()
        self.data_dict = gather_data_dict(
            self.scene_path,
            downscale_ratio=self.downscale_ratio,
            include_stereo=False,
            include_depth=False,
            include_wrist_cam=True,
            max_frames=-1
        )
        self.proprio_dict = gather_trajectory(self.scene_path, max_frames=-1)
        first_camera_serial = list(self.data_dict.keys())[0]
        canonical_timestamps = self.data_dict[first_camera_serial]['timestamps']
        if self.num_frames > 0 and len(canonical_timestamps) > self.num_frames:
            time_skip_ratio = max(1, len(canonical_timestamps) // self.num_frames)
            canonical_timestamps = canonical_timestamps[::time_skip_ratio]
            if len(canonical_timestamps) > self.num_frames:
                canonical_timestamps = canonical_timestamps[:self.num_frames]
        self.T = len(canonical_timestamps)
        try:
            self.data_dict = filter_by_timestamps(self.data_dict, canonical_timestamps, self.scene_path, is_proprio=False)
            self.proprio_dict = filter_by_timestamps(self.proprio_dict, canonical_timestamps, self.scene_path, is_proprio=True)
        except RuntimeError as e:
            _print(f"Error during timestamp filtering: {e}")
            raise
        flipped_wrist_rgb = []
        for frame in self.data_dict[self.wrist_serial]['rgb']:
            frame = cv2.rotate(frame, cv2.ROTATE_180)
            flipped_wrist_rgb.append(frame)
        self.data_dict[self.wrist_serial]['rgb'] = flipped_wrist_rgb
        self.T = len(self.proprio_dict['joint_positions'])
        for camera_serial in self.data_dict:
            assert 'intrinsic' in self.data_dict[camera_serial], "intrinsic not found"
            self.data_dict[camera_serial]['measured_intrinsics'] = self.data_dict[camera_serial]['intrinsic'].copy()
            del self.data_dict[camera_serial]['intrinsic']
        robot_serial = get_robot_serial(self.scene_path)
        self.T_gripper_wrist = get_robot_transform(robot_serial=robot_serial)
        _print(f"Loaded data for {len(self.data_dict)} cameras, {self.T} frames (time taken: {time.time() - start:.2f}s)")
    
    def _compute_T_base_wrist(self, frame_idx):
        """Get base->wrist transform for a specific frame (includes vertical flip)."""
        T_gripper_base_quat = self.proprio_dict['gripper_pose'][frame_idx]
        T_gripper_base = transform_utils.convert_pose_quat2mat(T_gripper_base_quat[None])[0]
        T_base_gripper = np.linalg.inv(T_gripper_base)
        T_base_wrist = self.T_gripper_wrist @ T_base_gripper
        flip_transform = np.array([
            [-1,  0,  0,  0],
            [ 0, -1,  0,  0],
            [ 0,  0,  1,  0],
            [ 0,  0,  0,  1]
        ], dtype=np.float64)
        T_base_wrist_flipped = flip_transform @ T_base_wrist
        return T_base_wrist_flipped

    def _compute_frame_camera_loss(self, world_pts, optimized_T, depth_tensor, intrinsic, H, W, dedup_threshold):
        assert isinstance(world_pts, torch.Tensor)
        assert isinstance(optimized_T, torch.Tensor)
        assert isinstance(depth_tensor, torch.Tensor)
        assert isinstance(intrinsic, torch.Tensor)
        device = world_pts.device
        assert optimized_T.device == device
        assert depth_tensor.device == device
        assert intrinsic.device == device
        if not isinstance(H, torch.Tensor):
            H = torch.tensor(H, device=device, dtype=torch.float32)
        if not isinstance(W, torch.Tensor):
            W = torch.tensor(W, device=device, dtype=torch.float32)
        N_points = world_pts.shape[0]
        zero_loss = torch.tensor(0.0, device=device)
        zero_count = torch.tensor(0, device=device)
        ones = torch.ones((N_points, 1), device=device, dtype=torch.float32)
        pts_h = torch.cat([world_pts, ones], dim=1)
        pts_cam = torch.mm(pts_h, optimized_T.T)
        z = pts_cam[:, 2]
        valid_depth = z > 0
        if not torch.any(valid_depth):
            return zero_loss, zero_count
        pts_cam_valid = pts_cam[valid_depth]
        z_valid = z[valid_depth]
        proj_h = torch.mm(pts_cam_valid[:, :3], intrinsic.T)
        xy = proj_h[:, :2] / (proj_h[:, 2:3] + 1e-8)
        valid_bounds = (xy[:, 0] >= 0) & (xy[:, 0] < W) & (xy[:, 1] >= 0) & (xy[:, 1] < H)
        if not torch.any(valid_bounds):
            return zero_loss, zero_count
        xy_valid = xy[valid_bounds]
        z_final = z_valid[valid_bounds]
        if dedup_threshold > 0:
            unique_indices = deduplicate_coordinates(xy_valid, threshold_pixels=dedup_threshold)
            if len(unique_indices) == 0:
                return zero_loss, zero_count
            xy_final = xy_valid[unique_indices]
            z_final = z_final[unique_indices]
        else:
            xy_final = xy_valid
        sampled_depth = sample_depth_with_grid_sample(depth_tensor, xy_final)
        d_pred = z_final
        d_gt = sampled_depth
        valid_depth_range = (d_gt >= MIN_DEPTH_M) & (d_gt <= MAX_DEPTH_M)
        if not torch.any(valid_depth_range):
            return zero_loss, zero_count
        d_pred_filtered = d_pred[valid_depth_range]
        d_gt_filtered = d_gt[valid_depth_range]
        diff = torch.abs(d_gt_filtered - d_pred_filtered)
        frame_camera_loss = diff.mean()
        if torch.isfinite(frame_camera_loss):
            return frame_camera_loss, torch.tensor(d_pred_filtered.shape[0], device=device)
        else:
            return zero_loss, zero_count

    def _optimize_all_cameras_jointly(self, camera_data_dict, initial_extrinsics_dict,
                                      max_iterations, learning_rate, translation_scale,
                                      rotation_scale, dedup_threshold=0.1, optimizer_type="adam"):
        ext_camera_serials = [s for s in camera_data_dict.keys() if s != self.wrist_serial]
        n_ext_cameras = len(ext_camera_serials)
        translation_params_norm = torch.zeros(n_ext_cameras, 3, device=self.device, dtype=torch.float32, requires_grad=True)
        rotation_params_norm = torch.zeros(n_ext_cameras, 3, device=self.device, dtype=torch.float32, requires_grad=True)
        if optimizer_type.lower() == "lbfgs":
            optimizer = torch.optim.LBFGS([translation_params_norm, rotation_params_norm],
                lr=learning_rate, max_iter=3, tolerance_grad=1e-5, tolerance_change=1e-7)
        else:
            optimizer = torch.optim.Adam([translation_params_norm, rotation_params_norm],
                lr=learning_rate, eps=1e-6, weight_decay=0.0)
        extrinsic_init_tensor = torch.zeros((n_ext_cameras, 4, 4), dtype=torch.float32, device=self.device)
        camera_to_idx = {cam_serial: i for i, cam_serial in enumerate(ext_camera_serials)}
        for i, cam_serial in enumerate(ext_camera_serials):
            extrinsic_init_tensor[i] = torch.as_tensor(
                initial_extrinsics_dict[cam_serial], dtype=torch.float32, device=self.device
            )
        all_world_points = []
        for frame_idx in range(self.num_frames):
            joint_positions = self.proprio_dict['joint_positions'][frame_idx]
            gripper_positions = self.proprio_dict['gripper_positions'][frame_idx]
            fk_result = self.robot_renderer._get_forward_kinematics(joint_positions, gripper_positions)
            world_pts = self.robot_renderer._get_world_points(fk_result)
            assert world_pts.shape[0] > 0
            all_world_points.append(world_pts)
        all_world_points = torch.stack(all_world_points, dim=0)
        first_cam_serial = ext_camera_serials[0]
        first_depth = camera_data_dict[first_cam_serial]['depth'][0]
        H, W = first_depth.shape
        frame_groups = {}
        for frame_idx in range(all_world_points.shape[0]):
            frame_groups[frame_idx] = []
            for cam_serial in ext_camera_serials:
                camera_frame_data = camera_data_dict[cam_serial]
                depth_clean = camera_frame_data['depth'][frame_idx].copy()
                depth_tensor = torch.as_tensor(depth_clean, dtype=torch.float32, device=self.device)
                intrinsic_tensor = torch.as_tensor(camera_frame_data['intrinsic'], dtype=torch.float32, device=self.device)
                cam_idx = camera_to_idx[cam_serial]
                frame_groups[frame_idx].append((cam_idx, depth_tensor, intrinsic_tensor))
        total_points = all_world_points.shape[1]
        _print(f"    Batched {all_world_points.shape[0]} frames, {len(ext_camera_serials)} cameras, {total_points} robot points per frame")
        _print(f"    Using {optimizer_type.upper()} optimizer")
        _print(f"    Minimum robot points threshold: {self.min_robot_points}")
        total_valid_points = 0
        per_camera_losses_history = {cam_serial: [] for cam_serial in ext_camera_serials}
        final_per_camera_point_counts = {cam_serial: [] for cam_serial in ext_camera_serials}

        def compute_loss_vectorized():
            nonlocal total_valid_points, per_camera_losses_history, final_per_camera_point_counts
            optimizer.zero_grad()
            translation_params = translation_params_norm * translation_scale
            rotation_params = rotation_params_norm * rotation_scale
            optimized_extrinsics = torch.zeros_like(extrinsic_init_tensor)
            for i in range(n_ext_cameras):
                pose_6dof = torch.cat([translation_params[i], rotation_params[i]])
                delta_T = pose_6dof_to_matrix(pose_6dof)
                optimized_extrinsics[i] = delta_T @ extrinsic_init_tensor[i]
            all_losses = []
            total_valid_points = 0
            frame_camera_point_counts = {}
            per_camera_losses_this_iter = {cam_serial: [] for cam_serial in ext_camera_serials}
            for frame_idx, camera_data_list in frame_groups.items():
                world_pts = all_world_points[frame_idx]
                for cam_idx, depth_tensor, intrinsic in camera_data_list:
                    optimized_T = optimized_extrinsics[cam_idx]
                    frame_camera_loss, valid_points_count_tensor = self._compute_frame_camera_loss(
                        world_pts, optimized_T, depth_tensor, intrinsic, H, W, dedup_threshold
                    )
                    valid_points_count = valid_points_count_tensor.item()
                    cam_serial = ext_camera_serials[cam_idx]
                    frame_camera_point_counts[(frame_idx, cam_serial)] = valid_points_count
                    if valid_points_count > 0:
                        all_losses.append(frame_camera_loss)
                        total_valid_points += valid_points_count
                        per_camera_losses_this_iter[cam_serial].append(frame_camera_loss.item())
            for (frame_idx, cam_serial), point_count in frame_camera_point_counts.items():
                if point_count < self.min_robot_points:
                    raise RobotVisibilityError(
                        f"Camera {cam_serial} at frame {frame_idx} sees insufficient robot points during optimization. "
                        f"Found {point_count} robot points, but require at least {self.min_robot_points} points.",
                        error_type="insufficient_robot_points_per_frame_camera"
                    )
            if all_losses:
                data_loss = torch.stack(all_losses).mean()
            else:
                data_loss = torch.tensor(1e6, device=self.device, dtype=torch.float32)
            total_loss = data_loss
            for cam_serial in ext_camera_serials:
                if per_camera_losses_this_iter[cam_serial]:
                    avg_loss = np.mean(per_camera_losses_this_iter[cam_serial])
                    per_camera_losses_history[cam_serial].append(avg_loss)
                else:
                    per_camera_losses_history[cam_serial].append(0.0)
            for cam_serial in ext_camera_serials:
                final_per_camera_point_counts[cam_serial] = []
                for frame_idx in range(self.num_frames):
                    point_count = frame_camera_point_counts.get((frame_idx, cam_serial), 0)
                    final_per_camera_point_counts[cam_serial].append(point_count)
            if torch.isfinite(total_loss):
                total_loss.backward()
            else:
                _print("    Warning: Loss is not finite!")
            return total_loss

        initial_loss = compute_loss_vectorized()
        loss_history = []
        for iteration in range(max_iterations):
            if not (torch.isfinite(translation_params_norm).all() and torch.isfinite(rotation_params_norm).all()):
                _print(f"    iter {iteration:03d} | NaN/inf detected in parameters, stopping")
                break
            loss_value = optimizer.step(compute_loss_vectorized)
            current_loss = loss_value.item() if torch.isfinite(loss_value) else float('inf')
            loss_history.append(current_loss)
            if iteration % 50 == 0 or iteration < 5 or iteration == max_iterations - 1:
                with torch.no_grad():
                    total_trans_diff_cm = 0.0
                    total_rot_diff_deg = 0.0
                    for i, cam_serial in enumerate(ext_camera_serials):
                        translation_params = translation_params_norm[i].detach() * translation_scale
                        rotation_params = rotation_params_norm[i].detach() * rotation_scale
                        pose_6dof = torch.cat([translation_params, rotation_params])
                        delta_T = pose_6dof_to_matrix(pose_6dof)
                        optimized_T = delta_T @ extrinsic_init_tensor[i]
                        trans_diff_cm = torch.linalg.norm(optimized_T[:3, 3] - extrinsic_init_tensor[i][:3, 3]).item() * 100
                        R_rel = optimized_T[:3, :3] @ extrinsic_init_tensor[i][:3, :3].T
                        rot_trace = torch.clamp((torch.trace(R_rel) - 1) / 2, -1.0, 1.0)
                        rot_diff_deg = torch.acos(rot_trace).item() * 180 / np.pi
                        total_trans_diff_cm += trans_diff_cm
                        total_rot_diff_deg += rot_diff_deg
                    avg_trans_diff_cm = total_trans_diff_cm / n_ext_cameras
                    avg_rot_diff_deg = total_rot_diff_deg / n_ext_cameras
                avg_points_per_frame = total_valid_points / (self.num_frames * n_ext_cameras)
                robot_status = f"ROBOT_VISIBLE ({avg_points_per_frame:.1f} avg per camera-frame)"
                _print(f"    iter {iteration:03d} | loss {current_loss:.6f} | avg_trans {avg_trans_diff_cm:.2f}cm | avg_rot {avg_rot_diff_deg:.2f}deg | {robot_status}")

        optimized_extrinsics_dict = {}
        for i, cam_serial in enumerate(ext_camera_serials):
            final_translation = translation_params_norm[i].detach() * translation_scale
            final_rotation = rotation_params_norm[i].detach() * rotation_scale
            final_pose_6dof = torch.cat([final_translation, final_rotation])
            optimized_extrinsic = (pose_6dof_to_matrix(final_pose_6dof) @ extrinsic_init_tensor[i]).cpu().numpy()
            optimized_extrinsics_dict[cam_serial] = optimized_extrinsic
            extrinsic_init_np = extrinsic_init_tensor[i].cpu().numpy()
            trans_change_cm = np.linalg.norm(optimized_extrinsic[:3, 3] - extrinsic_init_np[:3, 3]) * 100
            R_rel_np = optimized_extrinsic[:3, :3] @ extrinsic_init_np[:3, :3].T
            rot_change_deg = np.arccos(np.clip((np.trace(R_rel_np) - 1) / 2, -1, 1)) * 180 / np.pi
            _print(f"    Camera {cam_serial} final change: {trans_change_cm:.2f}cm translation, {rot_change_deg:.2f}deg rotation")

        optimization_metrics = {
            "initial_loss": float(initial_loss.item()),
            "final_loss": float(loss_history[-1]),
            "min_robot_points_threshold": self.min_robot_points
        }
        all_point_counts = []
        frames_with_sufficient_visibility = 0
        for frame_idx in range(self.num_frames):
            frame_has_sufficient_visibility = True
            for cam_serial in ext_camera_serials:
                point_count = final_per_camera_point_counts[cam_serial][frame_idx] if frame_idx < len(final_per_camera_point_counts[cam_serial]) else 0
                all_point_counts.append(point_count)
                if point_count < self.min_robot_points:
                    frame_has_sufficient_visibility = False
            if frame_has_sufficient_visibility:
                frames_with_sufficient_visibility += 1
        assert all_point_counts
        optimization_metrics.update({
            "average_robot_points": float(np.mean(all_point_counts)),
            "max_robot_points": int(np.max(all_point_counts)),
            "min_robot_points": int(np.min(all_point_counts)),
            "frames_with_sufficient_visibility": frames_with_sufficient_visibility
        })
        for cam_serial in ext_camera_serials:
            camera_type = "wrist" if cam_serial == self.wrist_serial else "ext"
            prefix = f"{cam_serial}+{camera_type}_"
            camera_point_counts = final_per_camera_point_counts[cam_serial]
            camera_loss_history = per_camera_losses_history[cam_serial]
            assert camera_point_counts
            assert camera_loss_history
            optimization_metrics[f"{prefix}average_robot_points"] = float(np.mean(camera_point_counts))
            optimization_metrics[f"{prefix}max_robot_points"] = int(np.max(camera_point_counts))
            optimization_metrics[f"{prefix}min_robot_points"] = int(np.min(camera_point_counts))
            optimization_metrics[f"{prefix}frames_with_sufficient_visibility"] = sum(1 for count in camera_point_counts if count >= self.min_robot_points)
            optimization_metrics[f"{prefix}initial_loss"] = float(camera_loss_history[0])
            optimization_metrics[f"{prefix}final_loss"] = float(camera_loss_history[-1])
        return optimized_extrinsics_dict, optimization_metrics

    def add_depth_valid_masks(self):
        """Add depth valid masks following legacy single-stage pipeline pattern, type-dependent"""
        for camera_serial in self.data_dict:
            keys_to_process = list(self.data_dict[camera_serial].keys())
            for key in keys_to_process:
                if key.endswith('_depth'):
                    depth_frames = self.data_dict[camera_serial][key]
                    depth_valid_mask = np.isfinite(depth_frames) & (depth_frames >= MIN_DEPTH_M) & (depth_frames <= MAX_DEPTH_M)
                    self.data_dict[camera_serial][f'{key}_valid_mask'] = depth_valid_mask

    def _prepare_vggt_input(self):
        """Prepare VGGT input ordering with external camera first."""
        aggregator_images = []
        aggregator_cam_ids = []
        aggregator_frame_idx = []
        first_ext_serial = self.ext_serials[0]
        for i in range(len(self.data_dict[first_ext_serial]['rgb'])):
            aggregator_images.append(self.data_dict[first_ext_serial]['rgb'][i])
            aggregator_cam_ids.append(first_ext_serial)
            aggregator_frame_idx.append(i)
        for i in range(len(self.data_dict[self.wrist_serial]['rgb'])):
            aggregator_images.append(self.data_dict[self.wrist_serial]['rgb'][i])
            aggregator_cam_ids.append(self.wrist_serial)
            aggregator_frame_idx.append(i)
        for cam_serial in self.ext_serials[1:]:
            for i in range(len(self.data_dict[cam_serial]['rgb'])):
                aggregator_images.append(self.data_dict[cam_serial]['rgb'][i])
                aggregator_cam_ids.append(cam_serial)
                aggregator_frame_idx.append(i)
        return aggregator_images, aggregator_cam_ids, aggregator_frame_idx
    
    def _convert_vggt_to_base_frame(self, vggt_extrinsics, aggregator_cam_ids, aggregator_frame_idx):
        result = {}
        wrist_indices = [idx for idx, cser in enumerate(aggregator_cam_ids) if cser == self.wrist_serial]
        assert len(wrist_indices) > 0
        T_base_ext0_estimates = []
        for wrist_idx in wrist_indices:
            frame_idx = aggregator_frame_idx[wrist_idx]
            T_base_wrist_t = self._compute_T_base_wrist(frame_idx)
            T_ext0_wrist_t = vggt_extrinsics[wrist_idx]
            T_wrist_t_ext0 = np.linalg.inv(T_ext0_wrist_t)
            T_base_ext0_estimate = T_wrist_t_ext0 @ T_base_wrist_t
            T_base_ext0_estimates.append(T_base_ext0_estimate)
        if len(T_base_ext0_estimates) > 1:
            T_base_ext0_avg = transform_utils.average_poses(T_base_ext0_estimates)
        else:
            T_base_ext0_avg = T_base_ext0_estimates[0]
        result[self.ext_serials[0]] = T_base_ext0_avg
        other_ext_indices = {cam_serial: [] for cam_serial in self.ext_serials if cam_serial != self.ext_serials[0]}
        for idx, cam_serial in enumerate(aggregator_cam_ids):
            if cam_serial != self.wrist_serial and cam_serial != self.ext_serials[0]:
                assert cam_serial in self.ext_serials
                other_ext_indices[cam_serial].append(idx)
        for cam_serial, indices in other_ext_indices.items():
            T_ext0_cami_estimates = [vggt_extrinsics[idx] for idx in indices]
            if len(T_ext0_cami_estimates) > 1:
                T_ext0_cami_avg = transform_utils.average_poses(T_ext0_cami_estimates)
            else:
                T_ext0_cami_avg = T_ext0_cami_estimates[0]
            T_base_cam_i = T_ext0_cami_avg @ T_base_ext0_avg
            result[cam_serial] = T_base_cam_i
        return result

    def _filter_frames_by_robot_visibility(self, extrinsics_dict):
        start = time.time()
        valid_frame_indices = []
        for frame_idx in range(self.T):
            frame_valid = True
            joint_positions = self.proprio_dict['joint_positions'][frame_idx]
            gripper_positions = self.proprio_dict['gripper_positions'][frame_idx]
            fk_result = self.robot_renderer._get_forward_kinematics(joint_positions, gripper_positions)
            world_pts = self.robot_renderer._get_world_points(fk_result)
            for camera_serial in self.data_dict:
                if camera_serial == self.wrist_serial:
                    continue
                if camera_serial not in extrinsics_dict:
                    continue
                extrinsic = extrinsics_dict[camera_serial]
                depth_frame = self.data_dict[camera_serial]['stereo_depth'][frame_idx]
                intrinsic = self.data_dict[camera_serial]['stereo_intrinsics']
                world_pts_tensor = torch.as_tensor(world_pts, dtype=torch.float32, device=self.device)
                extrinsic_tensor = torch.as_tensor(extrinsic, dtype=torch.float32, device=self.device)
                depth_tensor = torch.as_tensor(depth_frame, dtype=torch.float32, device=self.device)
                intrinsic_tensor = torch.as_tensor(intrinsic, dtype=torch.float32, device=self.device)
                H, W = depth_frame.shape
                _, valid_points_count_tensor = self._compute_frame_camera_loss(
                    world_pts_tensor, extrinsic_tensor, depth_tensor, intrinsic_tensor, H, W, dedup_threshold=0.1
                )
                valid_points_count = valid_points_count_tensor.item()
                if valid_points_count < self.min_robot_points:
                    _print(f"    Frame {frame_idx}: Camera {camera_serial} sees only {valid_points_count} robot points (< {self.min_robot_points}), removing frame")
                    frame_valid = False
                    break
            if frame_valid:
                valid_frame_indices.append(frame_idx)
        _print(f"Kept {len(valid_frame_indices)}/{self.T} frames after robot visibility filtering (time taken: {time.time() - start:.2f}s)")
        if len(valid_frame_indices) == 0:
            raise RobotVisibilityError(
                f"No frames have sufficient robot visibility. All {self.T} frames were filtered out because "
                f"at least one camera in each frame sees fewer than {self.min_robot_points} robot points.",
                error_type="no_valid_frames_after_filtering"
            )
        return valid_frame_indices

    def _update_data_for_valid_frames(self, valid_frame_indices):
        """Update all data structures to only include valid frames."""
        if len(valid_frame_indices) == self.T:
            return
        _print(f"Updating data structures for {len(valid_frame_indices)} valid frames...")
        for key in self.proprio_dict:
            if isinstance(self.proprio_dict[key], list):
                self.proprio_dict[key] = [self.proprio_dict[key][i] for i in valid_frame_indices]
            elif isinstance(self.proprio_dict[key], np.ndarray) and len(self.proprio_dict[key]) == self.T:
                self.proprio_dict[key] = self.proprio_dict[key][valid_frame_indices]
        for camera_serial in self.data_dict:
            for key in self.data_dict[camera_serial]:
                if isinstance(self.data_dict[camera_serial][key], list) and len(self.data_dict[camera_serial][key]) == self.T:
                    self.data_dict[camera_serial][key] = [self.data_dict[camera_serial][key][i] for i in valid_frame_indices]
                elif isinstance(self.data_dict[camera_serial][key], np.ndarray) and len(self.data_dict[camera_serial][key]) == self.T:
                    self.data_dict[camera_serial][key] = self.data_dict[camera_serial][key][valid_frame_indices]
        self.T = len(valid_frame_indices)
        self.num_frames = min(self.num_frames, self.T)

    def vggt_estimation(self):
        """Estimate initial extrinsics and intrinsics using VGGT."""
        assert self.vggt_forward is not None, "No VGGT model provided"
        start = time.time()
        aggregator_images, aggregator_cam_ids, aggregator_frame_idx = self._prepare_vggt_input()
        if not aggregator_images:
            raise ValueError("No images prepared for VGGT input")
        with stage_vggt_images(aggregator_images) as temp_image_paths:
            vggt_result = self.vggt_forward(temp_image_paths)
        vggt_extrinsics_4x4 = vggt_result['extrinsics_4x4']  # [N, 4, 4]
        vggt_intrinsics = vggt_result['intrinsics']  # [N, 3, 3]
        base_frame_extrinsics = self._convert_vggt_to_base_frame(
            vggt_extrinsics_4x4, aggregator_cam_ids, aggregator_frame_idx,
        )
        for idx, cam_serial in enumerate(aggregator_cam_ids):
            if cam_serial == self.wrist_serial:
                continue
            assert cam_serial in self.data_dict
            assert cam_serial in base_frame_extrinsics
            self.data_dict[cam_serial]['vggt_extrinsics'] = base_frame_extrinsics[cam_serial]
            self.data_dict[cam_serial]['vggt_intrinsics'] = vggt_intrinsics[idx]
        _print(f"VGGT estimation completed for {len(base_frame_extrinsics)} cameras (time taken: {time.time() - start:.2f}s)")
        valid_frame_indices = self._filter_frames_by_robot_visibility(base_frame_extrinsics)
        self._update_data_for_valid_frames(valid_frame_indices)

    def _collect_optimizer_inputs(self, depth_key, intrinsics_key, extrinsics_key):
        camera_data_dict = {}
        initial_extrinsics_dict = {}
        for camera_serial in self.data_dict:
            if camera_serial == self.wrist_serial:
                continue
            camera_data_dict[camera_serial] = {
                'rgb': self.data_dict[camera_serial]['rgb'],
                'depth': self.data_dict[camera_serial][depth_key],
                'depth_valid_mask': self.data_dict[camera_serial][f'{depth_key}_valid_mask'],
                'intrinsic': self.data_dict[camera_serial][intrinsics_key],
            }
            initial_extrinsics_dict[camera_serial] = self.data_dict[camera_serial][extrinsics_key]
        return camera_data_dict, initial_extrinsics_dict

    def optimize_extrinsics(self, max_iterations=100, learning_rate=0.001,
                            translation_scale=0.01, rotation_scale=np.pi/180.0,
                            dedup_threshold=0.5, optimizer_type="adam"):
        start = time.time()
        _print(f"  Using optimizer: {optimizer_type.upper()}")
        assert self.robot_renderer is not None
        camera_data_dict, initial_extrinsics_dict = self._collect_optimizer_inputs(
            depth_key='stereo_depth',
            intrinsics_key='stereo_intrinsics',
            extrinsics_key='vggt_extrinsics',
        )
        optimized_extrinsics_dict, optimization_metrics = self._optimize_all_cameras_jointly(
            camera_data_dict=camera_data_dict,
            initial_extrinsics_dict=initial_extrinsics_dict,
            max_iterations=max_iterations,
            learning_rate=learning_rate,
            translation_scale=translation_scale,
            rotation_scale=rotation_scale,
            dedup_threshold=dedup_threshold,
            optimizer_type=optimizer_type
        )
        for camera_serial, optimized_extrinsic in optimized_extrinsics_dict.items():
            self.data_dict[camera_serial]['optimized_extrinsics'] = optimized_extrinsic
        self.optimization_metrics = optimization_metrics
        _print(f"Joint extrinsics optimization completed for {len(optimized_extrinsics_dict)} cameras (time taken: {time.time() - start:.2f}s)")

    def process(self, output_dir=None, max_iterations=100, learning_rate=0.001,
                translation_scale=0.01, rotation_scale=np.pi/180.0,
                dedup_threshold=1.0, optimizer_type="adam", min_robot_points=None):
        if min_robot_points is not None:
            self.min_robot_points = min_robot_points
        self._load_scene_data()
        if self.T < self.num_frames:
            raise ValueError(f"too few frames in scene {self.scene_path}, T={self.T}, num_frames={self.num_frames}, skipping")
        if output_dir is None:
            raise ValueError("output_dir must be provided to load precomputed depth")
        load_precomputed_depth(
            output_dir=output_dir,
            uuid=self.uuid,
            data_dict=self.data_dict,
            wrist_serial=self.wrist_serial,
            log_fn=_print,
        )
        self.vggt_estimation()
        self.add_depth_valid_masks()
        self.optimize_extrinsics(
            max_iterations=max_iterations,
            learning_rate=learning_rate,
            translation_scale=translation_scale,
            rotation_scale=rotation_scale,
            dedup_threshold=dedup_threshold,
            optimizer_type=optimizer_type,
        )
        if output_dir is not None:
            write_camera_results(
                output_dir=output_dir,
                uuid=self.uuid,
                scene_path=self.scene_path,
                data_dict=self.data_dict,
                wrist_serial=self.wrist_serial,
                optimization_metrics=self.optimization_metrics,
                error_info=None,
                log_fn=_print,
            )
        return None

################################################################################
# Helper functions
################################################################################

def determine_error_stage():
    import traceback
    import sys
    exc_type, exc_value, exc_traceback = sys.exc_info()
    if exc_traceback is not None:
        stack = traceback.extract_tb(exc_traceback)
        stack_str = str(stack)
    else:
        stack_str = str(traceback.extract_stack())
    if '_optimize_all_cameras_jointly' in stack_str or 'optimize_extrinsics' in stack_str:
        return "extrinsics_optimization"
    elif '_filter_frames_by_robot_visibility' in stack_str or 'vggt_estimation' in stack_str:
        return "vggt_estimation"
    elif '_load_scene_data' in stack_str:
        return "data_loading"
    else:
        return "unknown"

################################################################################
# Main function
################################################################################

def process_single_scene(scene_path, output_dir, vggt_forward, robot_renderer,
                        num_frames, downscale_ratio, min_robot_points,
                        max_iterations, lr, translation_scale, rotation_scale, dedup_threshold, optimizer,
                        debug):
    """Process a single scene for extrinsics optimization."""
    try:
        uuid = get_uuid(scene_path)
        if output_dir is None:
            raise ValueError("output_dir must be provided to load precomputed depth")
        depth_exists, h5_path, error_msg = check_precomputed_depth_exists(scene_path, output_dir, uuid)
        if not depth_exists:
            raise FileNotFoundError(error_msg)
        _print(f"Verified precomputed depth file exists: {h5_path}")
        if output_dir is not None:
            results_exist, json_path, error_msg = check_camera_results_exist(scene_path, output_dir, uuid)
            if results_exist:
                _print(f"Camera results already exist and are readable: {json_path}, skipping scene")
                return True
        start = time.time()
        scene_optimizer = ExtrinsicsOptimizer(
            scene_path=scene_path,
            vggt_forward=vggt_forward,
            robot_renderer=robot_renderer,
            device=DEVICE,
            num_frames=num_frames,
            downscale_ratio=downscale_ratio,
            min_robot_points=min_robot_points,
        )
        scene_optimizer.process(
            output_dir=output_dir,
            max_iterations=max_iterations,
            learning_rate=lr,
            translation_scale=translation_scale,
            rotation_scale=rotation_scale,
            dedup_threshold=dedup_threshold,
            optimizer_type=optimizer
        )
        _print(f"Successfully processed scene: {scene_optimizer.uuid} (time taken: {time.time() - start:.2f}s)")
        return True
    except RobotVisibilityError as e:
        _print(f"Insufficient robot visibility for scene {scene_path}: {e}")
        if output_dir is not None:
            assert scene_optimizer is not None
            error_info = {
                "error_type": getattr(e, 'error_type', 'robot_visibility_error'),
                "error_message": str(e),
                "stage": determine_error_stage(),
                "min_robot_points_threshold": min_robot_points
            }
            write_camera_results(
                output_dir=output_dir,
                uuid=scene_optimizer.uuid,
                scene_path=scene_path,
                data_dict=scene_optimizer.data_dict,
                wrist_serial=scene_optimizer.wrist_serial,
                optimization_metrics=scene_optimizer.optimization_metrics,
                error_info=error_info,
                log_fn=_print,
            )
            _print(f"Saved partial results with robot visibility error for scene {scene_path}")
        return False
    except Exception as e:
        if 'Precomputed depth file not found' in str(e):
            _print(f"Precomputed depth file not found for scene {scene_path}, skipping")
            return False
        error_msg = f"Error processing scene {scene_path}: {e}"
        if debug:
            _print(f"{error_msg}")
            import traceback
            traceback.print_exc()
            raise
        else:
            _print(f"{error_msg}, skipping")
            return False

In [ ]:
print("All dependency modules written to disk. Ready to import.")

---
## Step 2 — Verify Imports

Import the core dependencies to catch any issues early.

In [ ]:
from real.gcs_utils import enforce_gcs_cache_policy
from real.real_utils import get_time_str
from real.droid_utils import gather_data_dict, get_uuid
print("real.* imports OK")

---
## Part 1: Compute Depth

Compute stereo depth for all available cameras in DROID episodes and store in dedicated H5 files.
Uses FoundationStereo for stereo depth estimation.

In [ ]:
import os
import numpy as np
import h5py
import random
from tqdm import tqdm

# FoundationStereo checkpoint paths (relative to REPO_ROOT set in Step 0)
DEPTH_ESTIMATOR_CKPT_PATH = os.path.join(
    REPO_ROOT, "checkpoints", "foundationstereo", "23-51-11", "model_best_bp2.pth"
)
DEPTH_ESTIMATOR_CFG_PATH = os.path.join(
    REPO_ROOT, "assets", "foundationstereo", "23-51-11", "cfg.yaml"
)

print("Depth estimator ckpt path:", DEPTH_ESTIMATOR_CKPT_PATH)
print("Depth estimator cfg path :", DEPTH_ESTIMATOR_CFG_PATH)

In [ ]:
def check_depth_file_exists(output_dir, uuid):
    """
    Check if depth file exists and has valid structure with write_complete flags.

    Args:
        output_dir (str): Output directory for depth files
        uuid (str): Episode UUID

    Returns:
        bool: True if valid depth file exists, False otherwise
    """
    h5_path = os.path.join(output_dir, "depth", f"{uuid}_depth.h5")

    if not os.path.exists(h5_path):
        return False

    with h5py.File(h5_path, 'r') as f:
        # Check metadata
        if 'metadata' not in f:
            raise ValueError(f"Depth file missing metadata group: {h5_path}")

        metadata = f['metadata']
        if 'write_complete' not in metadata.attrs or not metadata.attrs['write_complete']:
            raise ValueError(f"Depth file metadata incomplete: {h5_path}")

        # Check each camera group
        camera_groups = [key for key in f.keys() if key != 'metadata']
        if not camera_groups:
            raise ValueError(f"Depth file missing camera groups: {h5_path}")

        for camera_group_name in camera_groups:
            camera_group = f[camera_group_name]

            # Check required datasets
            required_keys = ['depth', 'timestamps']
            for key in required_keys:
                if key not in camera_group:
                    raise ValueError(f"Depth file missing {key} in {camera_group_name}: {h5_path}")
                if 'write_complete' not in camera_group[key].attrs or not camera_group[key].attrs['write_complete']:
                    raise ValueError(f"Depth file {key} incomplete in {camera_group_name}: {h5_path}")

    return True

In [ ]:
def compute_depth_for_scene(
    scene_path,
    output_dir,
    depth_estimator,
    downscale_ratio=1.0,
    batch_size=32,
):
    """
    Compute stereo depth for all cameras in a single scene.

    Args:
        scene_path (str): Path to the scene directory
        output_dir (str): Output directory for depth files
        depth_estimator (DepthEstimator): Initialized depth estimator
        downscale_ratio (float): Downscale ratio for the images
        batch_size (int): Batch size for depth estimation

    Returns:
        bool: True if successful
    """
    # Get episode UUID
    uuid = get_uuid(scene_path)

    # Check if already processed
    if check_depth_file_exists(output_dir, uuid):
        print(f"[{get_time_str()}] Depth already computed for {uuid}, skipping")
        return True

    print(f"[{get_time_str()}] Processing scene: {uuid}")

    # Gather data with stereo frames for external cameras
    data_dict = gather_data_dict(
        scene_path,
        downscale_ratio=downscale_ratio,
        include_stereo=True,
        include_depth=False,  # We want to compute our own depth
    )
    # Create output H5 file
    depth_dir = os.path.join(output_dir, "depth")
    h5_path = os.path.join(depth_dir, f"{uuid}_depth.h5")
    os.makedirs(depth_dir, exist_ok=True)

    with h5py.File(h5_path, 'w') as f:
        # Create metadata group
        metadata_group = f.create_group('metadata')
        metadata_group.attrs['uuid'] = uuid
        metadata_group.attrs['camera_count'] = len(data_dict)

        total_cameras = len(data_dict)
        frame_count = None

        # Process each camera
        for i, (camera_serial, camera_data) in enumerate(data_dict.items()):
            print(f"[{get_time_str()}] Processing camera {i+1}/{total_cameras}: {camera_serial}")

            # Extract required data
            left_frames = camera_data['rgb']        # Left camera frames
            right_frames = camera_data['right_frames']  # Right camera frames
            timestamps = camera_data['timestamps']  # Timestamps
            intrinsic = camera_data['intrinsic']
            baseline = camera_data['baseline']
            if frame_count is None:
                frame_count = len(left_frames)

            # Set camera parameters for depth estimator
            depth_estimator.set_camera_params(baseline, intrinsic)

            # Compute depth using batch inference with smaller batch size for memory efficiency
            print(f"[{get_time_str()}] Computing depth for {len(left_frames)} frames...")
            depth_frames = depth_estimator.infer_depth_batch(left_frames, right_frames, batch_size=batch_size)

            # Convert depth to uint16 millimeters
            depth_mm = (depth_frames * 1000.0).astype(np.float32)
            depth_mm = np.clip(depth_mm, 0, 65535)  # Clip to uint16 range
            depth_uint16 = depth_mm.astype(np.uint16)

            camera_type = "ext"
            camera_group_name = f"{camera_serial}+{camera_type}"

            # Create camera group in H5 file with type suffix
            camera_group = f.create_group(camera_group_name)

            # Add camera metadata to group attributes
            camera_group.attrs['camera_serial'] = camera_serial
            camera_group.attrs['camera_type'] = camera_type

            # Store depth with high compression
            depth_dataset = camera_group.create_dataset(
                'depth',
                data=depth_uint16,
                compression='gzip',
                compression_opts=9,
                shuffle=True,
                chunks=True
            )
            depth_dataset.attrs['write_complete'] = True
            depth_dataset.attrs['units'] = 'millimeters'
            depth_dataset.attrs['dtype'] = 'uint16'

            # Store timestamps
            timestamps_dataset = camera_group.create_dataset(
                'timestamps',
                data=timestamps,
            )
            timestamps_dataset.attrs['write_complete'] = True
            timestamps_dataset.attrs['units'] = 'milliseconds'

            print(f"[{get_time_str()}] Saved depth data for camera {camera_group_name}: {depth_uint16.shape}")

        # Update metadata with frame count
        metadata_group.attrs['frame_count'] = frame_count
        metadata_group.attrs['write_complete'] = True

    print(f"[{get_time_str()}] Successfully saved depth data to {h5_path}")
    return True

### Compute Depth — Configuration

Edit the variables below, then run the next cells to execute.

In [ ]:
# ============================================================
# Configuration — edit these values
# ============================================================
INPUT_FILE        = "REPLACE_ME.txt"    # Path to txt file with one scene path per line
OUTPUT_DIR        = "REPLACE_ME_output" # Output directory for depth H5 files
RANK              = 0                   # Process rank (0-indexed)
WORLD_SIZE        = 1                   # Total number of parallel workers
DOWNSCALE_RATIO   = 0.5                 # Image downscale ratio (raw is 1280x720)
BATCH_SIZE        = 12                  # Batch size (>12 may cause cudnn error with AMP)
FOUNDATION_STEREO_CKPT = DEPTH_ESTIMATOR_CKPT_PATH   # FoundationStereo checkpoint
FOUNDATION_STEREO_CFG  = DEPTH_ESTIMATOR_CFG_PATH    # FoundationStereo cfg.yaml
ALLOW_GCS_STREAMING    = False          # Set True to stream from GCS without local cache

In [ ]:
# Read and partition scene paths
with open(INPUT_FILE, 'r') as f:
    all_paths = [line.strip() for line in f if line.strip()]

enforce_gcs_cache_policy(
    all_paths,
    stage_name="compute_depth",
    require_cache=True,
    allow_streaming=ALLOW_GCS_STREAMING,
)

# Shuffle with fixed seed for reproducibility and load balancing
random.seed(42)
random.shuffle(all_paths)

# Validate distributed args
if WORLD_SIZE <= 0:
    raise ValueError(f"WORLD_SIZE must be >= 1, got {WORLD_SIZE}")
if not (0 <= RANK < WORLD_SIZE):
    raise ValueError(f"RANK must be in [0, WORLD_SIZE), got rank={RANK}, world_size={WORLD_SIZE}")

paths_to_process = all_paths[RANK::WORLD_SIZE]
print(f"[{get_time_str()}] Rank {RANK}/{WORLD_SIZE}: Processing {len(paths_to_process)}/{len(all_paths)} scenes")

In [ ]:
# Import and initialize DepthEstimator
# NOTE: DepthEstimator lives in depth_estimator.py at the repo root.
# It internally adds third_party/FoundationStereo to sys.path.
import sys, os
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Write depth_estimator.py inline so it is importable
# (It references third_party/FoundationStereo which must be present at REPO_ROOT)
print(f"[{get_time_str()}] Initializing depth estimator...")
if not os.path.exists(FOUNDATION_STEREO_CKPT):
    raise FileNotFoundError(f"FoundationStereo checkpoint not found: {FOUNDATION_STEREO_CKPT}")
if not os.path.exists(FOUNDATION_STEREO_CFG):
    raise FileNotFoundError(f"FoundationStereo cfg not found: {FOUNDATION_STEREO_CFG}")

from depth_estimator import DepthEstimator
depth_estimator = DepthEstimator(
    FOUNDATION_STEREO_CKPT,
    device='cuda',
    cfg_path=FOUNDATION_STEREO_CFG,
)
print(f"[{get_time_str()}] Depth estimator ready.")

In [ ]:
# Process all scenes
total_processed = 0
total_successful = 0

for scene_path in tqdm(paths_to_process, desc="Processing scenes"):
    total_processed += 1
    success = compute_depth_for_scene(
        scene_path,
        OUTPUT_DIR,
        depth_estimator,
        downscale_ratio=DOWNSCALE_RATIO,
        batch_size=BATCH_SIZE,
    )
    if success:
        total_successful += 1

print(f"\n[{get_time_str()}] Processing completed!")
print(f"  Total scenes processed: {total_processed}")
print(f"  Successful: {total_successful}")
print(f"  Success rate: {100*total_successful/total_processed:.1f}%")

---
## Part 2: Compute Extrinsics

Extrinsics optimization using VGGT initialization and robot mesh rendering.
Takes estimated extrinsics (from VGGT) and optimizes them using precomputed stereo depth (from FoundationStereo).

In [ ]:
import sys, os, random
import numpy as np
import torch
from tqdm import tqdm

# Import the extrinsics pipeline (depends on modules written in Step 1)
from real.extrinsics_pipeline import _print, DEVICE, URDF_PATH, process_single_scene
from compute_extrinsics_utils import RobotMeshRenderer
from real.vggt_forward import VGGTForwardPass
from real.gcs_utils import enforce_gcs_cache_policy

print(f"Device: {DEVICE}")
print(f"URDF path: {URDF_PATH}")

### Compute Extrinsics — Configuration

Edit the variables below, then run the remaining cells.

In [ ]:
# ============================================================
# Configuration — edit these values
# ============================================================
EXT_OUTPUT_DIR        = "REPLACE_ME_output"  # Output dir (same as compute_depth OUTPUT_DIR)
EXT_INPUT_FILE        = None                  # Path to txt with scene paths (or set SCENE)
SCENE                 = None                  # Single scene path (set None to use EXT_INPUT_FILE)
EXT_RANK              = 0
EXT_WORLD_SIZE        = 1
DEBUG                 = False

# Processing parameters
NUM_FRAMES            = 10
DOWNSCALE             = 0.5
MAX_ITERATIONS        = 2000
LR                    = 0.05
TRANSLATION_SCALE     = 0.01
ROTATION_SCALE        = np.deg2rad(0.05)     # radians (~0.05 degrees)
DEDUP_THRESHOLD       = 0.5
MIN_ROBOT_POINTS      = 1000
VGGT_MODEL_PATH       = os.path.join(REPO_ROOT, "checkpoints", "vggt", "model.pt")
OPTIMIZER             = "adam"               # "adam" or "lbfgs"
EXT_ALLOW_GCS_STREAMING = False

In [ ]:
# Prepare scene paths
assert SCENE or EXT_INPUT_FILE, "Either SCENE or EXT_INPUT_FILE must be provided"

if EXT_INPUT_FILE:
    with open(EXT_INPUT_FILE, 'r') as f:
        ext_all_paths = [line.strip() for line in f if line.strip()]

    random.seed(42)
    random.shuffle(ext_all_paths)

    if EXT_WORLD_SIZE <= 0:
        raise ValueError(f"EXT_WORLD_SIZE must be >= 1, got {EXT_WORLD_SIZE}")
    if not (0 <= EXT_RANK < EXT_WORLD_SIZE):
        raise ValueError(f"EXT_RANK must be in [0, EXT_WORLD_SIZE), got rank={EXT_RANK}, world_size={EXT_WORLD_SIZE}")
    ext_paths_to_process = ext_all_paths[EXT_RANK::EXT_WORLD_SIZE]
else:
    ext_all_paths = [SCENE]
    ext_paths_to_process = ext_all_paths

enforce_gcs_cache_policy(
    ext_all_paths,
    stage_name="compute_extrinsics",
    require_cache=True,
    allow_streaming=EXT_ALLOW_GCS_STREAMING,
)

_print(f"Rank {EXT_RANK}/{EXT_WORLD_SIZE}: Processing {len(ext_paths_to_process)}/{len(ext_all_paths)} scenes")

In [ ]:
# Initialize VGGT model and robot renderer
if len(ext_paths_to_process) == 0:
    _print(f"No scenes to process for rank {EXT_RANK}")
else:
    # Load VGGT
    from vggt.models.vggt import VGGT
    assert os.path.exists(VGGT_MODEL_PATH), f"VGGT model file not found: {VGGT_MODEL_PATH}"
    vggt_model = VGGT()
    vggt_model.load_state_dict(torch.load(VGGT_MODEL_PATH, map_location=DEVICE))
    vggt_model.to(DEVICE)
    vggt_model.eval()
    vggt_forward = VGGTForwardPass(vggt_model, device=DEVICE)
    _print(f"VGGT model loaded successfully from {VGGT_MODEL_PATH}")

    # Initialize robot renderer
    robot_renderer = RobotMeshRenderer(URDF_PATH, device=DEVICE, total_samples=25000)

In [ ]:
# Process all scenes
total_processed = 0
total_successful = 0
total_skipped = 0

for scene_path in tqdm(ext_paths_to_process, desc=f"Rank {EXT_RANK}/{EXT_WORLD_SIZE}"):
    total_processed += 1

    success = process_single_scene(
        scene_path=scene_path,
        output_dir=EXT_OUTPUT_DIR,
        vggt_forward=vggt_forward,
        robot_renderer=robot_renderer,
        num_frames=NUM_FRAMES,
        downscale_ratio=DOWNSCALE,
        min_robot_points=MIN_ROBOT_POINTS,
        max_iterations=MAX_ITERATIONS,
        lr=LR,
        translation_scale=TRANSLATION_SCALE,
        rotation_scale=ROTATION_SCALE,
        dedup_threshold=DEDUP_THRESHOLD,
        optimizer=OPTIMIZER,
        debug=DEBUG
    )

    if success:
        total_successful += 1
    else:
        total_skipped += 1

_print(f"\nProcessing completed!")
_print(f"  Total scenes processed: {total_processed}")
_print(f"  Successful: {total_successful}")
_print(f"  Skipped: {total_skipped}")
_print(f"  Success rate: {100*total_successful/total_processed:.1f}%")